In [1]:
# Predictor vs Random Effect DEG for 48 Mathys
# Predictor vs Random-Effect DEG (Mathys et al. 48)
# Standalone: loads DEG + models, computes correlations, writes Excel sheet

import os
import numpy as np
import pandas as pd
import joblib
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# ============================
# PATHS
# ============================
deg_dir = "/n/scratch/users/a/adm808/Revision/Clincal_batch_DE_Outputs_revision"
model_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"

SHEET_NAME = "Predictor_vs_RE_DEG_Mathys48"

EXCEL_OUT = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/"
    f"Correlation_Analyses_{SHEET_NAME}.xlsx"
)

os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

# ============================
# Compute correlations
# ============================
results = []

for ct in cell_types:
    print(f"\n=== {ct} ===")

    # ---- Load DEG table ----
    deg_path = os.path.join(deg_dir, f"poisson_DE_results_{ct}.csv")
    deg = pd.read_csv(deg_path)

    deg["gene"] = deg["gene"].astype(str).str.strip()
    deg = deg.sort_values("p_adj", ascending=True)
    deg["deg_rank"] = np.arange(1, len(deg) + 1)

    # ---- Load models across splits ----
    all_split_data = {}
    all_genes = set()

    for split in range(1, 6):
        model_path = os.path.join(
            model_dir, ct, f"split_{split}", "maximal_classifier.joblib"
        )
        model = joblib.load(model_path)

        feats = model.feature_names_in_
        imps = model.feature_importances_

        all_genes.update(feats)
        all_split_data[split] = dict(zip(feats, imps))

    # ---- Mean normalized importance per gene ----
    rows = []
    for gene in sorted(all_genes):
        norm_imps = []

        for split in range(1, 6):
            imp = all_split_data[split].get(gene, 0.0)
            split_imps = np.array(list(all_split_data[split].values()))
            denom = split_imps[split_imps > 0].sum()

            norm_imps.append(
                imp / denom if imp > 0 and denom > 0 else 0.0
            )

        rows.append({
            "gene": gene.strip(),
            "mean_importance": np.mean(norm_imps)
        })

    imp_df = pd.DataFrame(rows)
    imp_df["pred_rank"] = imp_df["mean_importance"].rank(ascending=False)

    # ---- Merge + correlate ----
    merged = deg.merge(imp_df, on="gene", how="inner")

    rho, p = spearmanr(merged["deg_rank"], merged["pred_rank"])

    print(f"Spearman rho = {rho:.3f}, p = {p:.2e}")

    results.append({
        "cell_type": ct,
        "spearman_rho": rho,
        "p_value": p,
        "n_genes": merged.shape[0]
    })

# ============================
# FDR correction
# ============================
summary = pd.DataFrame(results)
summary["p_value_fdr"] = multipletests(
    summary["p_value"], method="fdr_bh"
)[1]

# ============================
# Build wide-format table
# ============================
wide = pd.DataFrame(index=[
    "Spearman rho",
    "p-value",
    "FDR-adjusted p-value",
    "Number of genes"
])

for ct in cell_types:
    row = summary[summary["cell_type"] == ct].iloc[0]
    wide[ct] = [
        row["spearman_rho"],
        row["p_value"],
        row["p_value_fdr"],
        row["n_genes"],
    ]

# ============================
# Description block (top rows)
# ============================
description = pd.DataFrame({
    "": [
        "Analysis: Predictor vs Random-Effect DEG Correlation",
        "DEG source: Mathys et al. (48), random-effects Poisson differential expression",
        "Predictor: Mean normalized gene importance across 5 independent train–test splits",
        "Statistic: Spearman rank correlation between DEG rank (by FDR adjusted p-value since same dataset) and predictor rank",
        "Computed independently for each cell type",
        "",
    ]
})

# ============================
# Write Excel sheet (safe logic)
# ============================
if os.path.exists(EXCEL_OUT):
    with pd.ExcelWriter(
        EXCEL_OUT,
        engine="openpyxl",
        mode="a",
        if_sheet_exists="replace"
    ) as writer:

        description.to_excel(
            writer,
            sheet_name=SHEET_NAME,
            index=False,
            header=False,
            startrow=0
        )

        wide.to_excel(
            writer,
            sheet_name=SHEET_NAME,
            startrow=len(description) + 1
        )

else:
    with pd.ExcelWriter(
        EXCEL_OUT,
        engine="openpyxl",
        mode="w"
    ) as writer:

        description.to_excel(
            writer,
            sheet_name=SHEET_NAME,
            index=False,
            header=False,
            startrow=0
        )

        wide.to_excel(
            writer,
            sheet_name=SHEET_NAME,
            startrow=len(description) + 1
        )

print("\n✔ Sheet written:", SHEET_NAME)
print("✔ Workbook:", EXCEL_OUT)


=== Ast ===
Spearman rho = 0.269, p = 5.63e-51

=== Mic ===
Spearman rho = 0.480, p = 1.46e-84

=== In ===
Spearman rho = 0.041, p = 1.93e-03

=== Oli ===
Spearman rho = 0.277, p = 2.47e-20

=== Opc ===
Spearman rho = 0.297, p = 4.60e-80

=== Ex ===
Spearman rho = 0.120, p = 5.17e-26

✔ Sheet written: Predictor_vs_RE_DEG_Mathys48
✔ Workbook: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/Correlation_Analyses_Predictor_vs_RE_DEG_Mathys48.xlsx


In [2]:
# import os
# import shutil

# # ============================
# # SOURCE + DESTINATION
# # ============================
# SRC_DIR = "/n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed"
# DST_DIR = "/n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Renamed"

# os.makedirs(DST_DIR, exist_ok=True)

# # ============================
# # Explicit filename mapping
# # ============================
# file_map = {
#     "poisson_DE_results_PFC_Ast_COMBINED.csv": "poisson_DE_results_Ast.csv",
#     "poisson_DE_results_PFC_Ex_COMBINED.csv": "poisson_DE_results_Ex.csv",
#     "poisson_DE_results_PFC_In_COMBINED.csv": "poisson_DE_results_In.csv",
#     "poisson_DE_results_PFC_Mic.csv": "poisson_DE_results_Mic.csv",
#     "poisson_DE_results_PFC_Mic_COMBINED.csv": "poisson_DE_results_Mic_COMBINED_DUPLICATE.csv",
#     "poisson_DE_results_PFC_Oli_COMBINED.csv": "poisson_DE_results_Oli.csv",
#     "poisson_DE_results_PFC_Opc_COMBINED.csv": "poisson_DE_results_Opc.csv",
# }

# # ============================
# # Copy + rename
# # ============================
# for src_name, dst_name in file_map.items():
#     src_path = os.path.join(SRC_DIR, src_name)
#     dst_path = os.path.join(DST_DIR, dst_name)

#     if not os.path.exists(src_path):
#         print(f"❌ Missing: {src_name}")
#         continue

#     shutil.copy2(src_path, dst_path)
#     print(f"✅ Copied: {src_name} → {dst_name}")

# print("\nAll requested DEG files copied and renamed.")

In [3]:
# DEG Mathys 2019 vs DEG Mathys 2024
# Standalone: loads DEG tables, computes log2FC correlations, writes Excel sheet

import os
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# ============================
# PATHS
# ============================
deg2019_dir = "/n/scratch/users/a/adm808/Revision/Clincal_batch_DE_Outputs_revision"
deg2023_dir = "/n/scratch/users/a/adm808/Revision/New_Data_DEG_results"

SHEET_NAME = "DEG_Mathys2019_vs_Mathys2024_log2FC"

EXCEL_OUT = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/"
    f"Correlation_Analyses_{SHEET_NAME}.xlsx"
)

os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

# ============================
# Compute correlations
# ============================
results = []

for ct in cell_types:
    print(f"\n=== {ct} ===")

    deg2019 = pd.read_csv(os.path.join(deg2019_dir, f"poisson_DE_results_{ct}.csv"))
    deg2023 = pd.read_csv(os.path.join(deg2023_dir, f"poisson_DE_results_PFC_{ct}.csv"))

    deg2019["gene"] = deg2019["gene"].astype(str).str.strip()
    deg2023["gene"] = deg2023["gene"].astype(str).str.strip()

    deg2019 = deg2019.dropna(subset=["log2FC"])
    deg2023 = deg2023.dropna(subset=["log2FC"])

    merged = deg2019[["gene", "log2FC"]].merge(
        deg2023[["gene", "log2FC"]],
        on="gene",
        suffixes=("_2019", "_2024"),
        how="inner"
    )

    rho, p = spearmanr(
        merged["log2FC_2019"],
        merged["log2FC_2024"]
    )

    print(f"Spearman rho = {rho:.3f}, p = {p:.2e}, n = {merged.shape[0]}")

    results.append({
        "cell_type": ct,
        "spearman_rho": rho,
        "p_value": p,
        "n_genes": merged.shape[0]
    })

# ============================
# FDR correction
# ============================
summary = pd.DataFrame(results)

summary["p_value_fdr"] = multipletests(
    summary["p_value"], method="fdr_bh"
)[1]

# ============================
# Build wide-format table
# ============================
wide = pd.DataFrame(index=[
    "Spearman rho",
    "p-value",
    "FDR-adjusted p-value",
    "Number of genes"
])

for ct in cell_types:
    row = summary[summary["cell_type"] == ct].iloc[0]
    wide[ct] = [
        row["spearman_rho"],
        row["p_value"],
        row["p_value_fdr"],
        row["n_genes"],
    ]

# ============================
# Description block
# ============================
description = pd.DataFrame({
    "": [
        "Analysis: DEG–DEG log2FC Correlation",
        "DEG sources: Mathys et al. 2019 vs Mathys et al. 2024",
        "Statistic: Spearman correlation of per-gene log2 fold change",
        "Genes matched by symbol and filtered for non-missing log2FC",
        "Computed independently for each cell type",
        "",
    ]
})

# ============================
# Write Excel sheet (safe logic)
# ============================
if os.path.exists(EXCEL_OUT):
    with pd.ExcelWriter(
        EXCEL_OUT,
        engine="openpyxl",
        mode="a",
        if_sheet_exists="replace"
    ) as writer:

        description.to_excel(
            writer,
            sheet_name=SHEET_NAME,
            index=False,
            header=False,
            startrow=0
        )

        wide.to_excel(
            writer,
            sheet_name=SHEET_NAME,
            startrow=len(description) + 1
        )

else:
    with pd.ExcelWriter(
        EXCEL_OUT,
        engine="openpyxl",
        mode="w"
    ) as writer:

        description.to_excel(
            writer,
            sheet_name=SHEET_NAME,
            index=False,
            header=False,
            startrow=0
        )

        wide.to_excel(
            writer,
            sheet_name=SHEET_NAME,
            startrow=len(description) + 1
        )

print("\n✔ Sheet written:", SHEET_NAME)
print("✔ Workbook:", EXCEL_OUT)


=== Ast ===
Spearman rho = 0.367, p = 3.96e-116, n = 3640

=== Mic ===
Spearman rho = 0.336, p = 9.46e-62, n = 2295

=== In ===
Spearman rho = 0.331, p = 8.50e-154, n = 6018

=== Oli ===
Spearman rho = 0.466, p = 2.09e-148, n = 2751

=== Opc ===
Spearman rho = 0.304, p = 4.01e-101, n = 4717

=== Ex ===
Spearman rho = 0.550, p = 0.00e+00, n = 8076

✔ Sheet written: DEG_Mathys2019_vs_Mathys2024_log2FC
✔ Workbook: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/Correlation_Analyses_DEG_Mathys2019_vs_Mathys2024_log2FC.xlsx


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


In [4]:
# Mathys 48 Predictors vs Skene et al Predictors (rank vs rank)
# Standalone: loads models, computes rank correlations, writes Excel sheet

import os
import pandas as pd
import numpy as np
import joblib
from scipy.stats import spearmanr

# ============================
# PATHS
# ============================
mathys_model_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
skene_model_dir  = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_Skene_revision"

SHEET_NAME = "PredRank_Mathys48_vs_Skene"

EXCEL_OUT = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/"
    f"Correlation_Analyses_{SHEET_NAME}.xlsx"
)

os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

results = []

# ============================
# Loop per cell type
# ============================
for ct in cell_types:
    print(f"\n=== {ct} ===")

    # ---------- Load Mathys predictors ----------
    all_split_data_mathys = {}
    all_features_mathys = set()

    for split in range(1, 6):
        model_path = os.path.join(
            mathys_model_dir, ct, f"split_{split}", "maximal_classifier.joblib"
        )
        model = joblib.load(model_path)

        feats = model.feature_names_in_
        imps = model.feature_importances_

        all_features_mathys.update(feats)
        all_split_data_mathys[split] = dict(zip(feats, imps))

    rows_mathys = []
    for gene in sorted(all_features_mathys):
        norm_imps = []

        for split in range(1, 6):
            imp = all_split_data_mathys[split].get(gene, 0.0)
            split_imps = np.array(list(all_split_data_mathys[split].values()))
            denom = split_imps[split_imps > 0].sum()

            norm_imps.append(
                imp / denom if imp > 0 and denom > 0 else 0.0
            )

        rows_mathys.append({
            "gene": gene.strip(),
            "mean_importance_mathys": np.mean(norm_imps)
        })

    mathys_df = pd.DataFrame(rows_mathys)
    mathys_df["rank_mathys"] = mathys_df["mean_importance_mathys"].rank(ascending=False)

    # ---------- Load Skene predictors ----------
    all_split_data_skene = {}
    all_features_skene = set()

    for split in range(1, 6):
        model_path = os.path.join(
            skene_model_dir, ct, f"split_{split}", "maximal_classifier.joblib"
        )
        model = joblib.load(model_path)

        feats = model.feature_names_in_
        imps = model.feature_importances_

        all_features_skene.update(feats)
        all_split_data_skene[split] = dict(zip(feats, imps))

    rows_skene = []
    for gene in sorted(all_features_skene):
        norm_imps = []

        for split in range(1, 6):
            imp = all_split_data_skene[split].get(gene, 0.0)
            split_imps = np.array(list(all_split_data_skene[split].values()))
            denom = split_imps[split_imps > 0].sum()

            norm_imps.append(
                imp / denom if imp > 0 and denom > 0 else 0.0
            )

        rows_skene.append({
            "gene": gene.strip(),
            "mean_importance_skene": np.mean(norm_imps)
        })

    skene_df = pd.DataFrame(rows_skene)
    skene_df["rank_skene"] = skene_df["mean_importance_skene"].rank(ascending=False)

    # ---------- Merge + correlate ----------
    merged = mathys_df.merge(skene_df, on="gene", how="inner")

    rho, p = spearmanr(
        merged["rank_mathys"],
        merged["rank_skene"]
    )

    print(f"Spearman rho = {rho:.3f}, p = {p:.2e}, n = {merged.shape[0]}")

    results.append({
        "cell_type": ct,
        "spearman_rho": rho,
        "p_value": p,
        "n_genes": merged.shape[0]
    })

# ============================
# Build summary table
# ============================
summary = pd.DataFrame(results)

wide = pd.DataFrame(index=[
    "Spearman rho",
    "p-value",
    "Number of genes"
])

for ct in cell_types:
    row = summary[summary["cell_type"] == ct].iloc[0]
    wide[ct] = [
        row["spearman_rho"],
        row["p_value"],
        row["n_genes"],
    ]

# ============================
# Description block
# ============================
description = pd.DataFrame({
    "": [
        "Analysis: Predictor–Predictor Rank Correlation",
        "Predictors: Mathys et al. (48) vs Skene et al.",
        "Predictor score: Mean normalized feature importance across 5 splits",
        "Statistic: Spearman correlation of gene importance ranks",
        "Computed independently for each cell type",
        "",
    ]
})

# ============================
# Write Excel sheet (safe logic)
# ============================
if os.path.exists(EXCEL_OUT):
    with pd.ExcelWriter(
        EXCEL_OUT,
        engine="openpyxl",
        mode="a",
        if_sheet_exists="replace"
    ) as writer:

        description.to_excel(
            writer,
            sheet_name=SHEET_NAME,
            index=False,
            header=False,
            startrow=0
        )

        wide.to_excel(
            writer,
            sheet_name=SHEET_NAME,
            startrow=len(description) + 1
        )

else:
    with pd.ExcelWriter(
        EXCEL_OUT,
        engine="openpyxl",
        mode="w"
    ) as writer:

        description.to_excel(
            writer,
            sheet_name=SHEET_NAME,
            index=False,
            header=False,
            startrow=0
        )

        wide.to_excel(
            writer,
            sheet_name=SHEET_NAME,
            startrow=len(description) + 1
        )

print("\n✔ Sheet written:", SHEET_NAME)
print("✔ Workbook:", EXCEL_OUT)


=== Ast ===
Spearman rho = 0.182, p = 2.33e-22, n = 2814

=== Mic ===
Spearman rho = 0.442, p = 3.50e-64, n = 1318

=== In ===
Spearman rho = 0.078, p = 3.23e-09, n = 5704

=== Oli ===
Spearman rho = 0.440, p = 3.18e-49, n = 1014

=== Opc ===
Spearman rho = 0.458, p = 8.43e-191, n = 3684

=== Ex ===
Spearman rho = 0.345, p = 7.08e-210, n = 7553

✔ Sheet written: PredRank_Mathys48_vs_Skene
✔ Workbook: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/Correlation_Analyses_PredRank_Mathys48_vs_Skene.xlsx


In [5]:
# Mathys–Skene predictor overlap
# Predictor = nonzero importance in ≥2 splits
# Report recall of the smaller predictor set
# Writes results to Excel sheet (standalone workbook)

import os
import joblib
import pandas as pd

# ============================
# PATHS
# ============================
mathys_model_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
skene_model_dir  = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_Skene_revision"

SHEET_NAME = "PredOverlap_Mathys48_vs_Skene"

EXCEL_OUT = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/"
    f"Correlation_Analyses_{SHEET_NAME}.xlsx"
)

os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

summary_rows = []

# ============================
# Loop per cell type
# ============================
for ct in cell_types:
    print(f"\n=== {ct} ===")

    # ---------- Mathys predictors ----------
    gene_counts_mathys = {}

    for split in range(1, 6):
        model = joblib.load(
            os.path.join(mathys_model_dir, ct, f"split_{split}", "maximal_classifier.joblib")
        )
        for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_counts_mathys[gene] = gene_counts_mathys.get(gene, 0) + 1

    mathys_predictors = {g for g, c in gene_counts_mathys.items() if c >= 2}

    # ---------- Skene predictors ----------
    gene_counts_skene = {}

    for split in range(1, 6):
        model = joblib.load(
            os.path.join(skene_model_dir, ct, f"split_{split}", "maximal_classifier.joblib")
        )
        for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_counts_skene[gene] = gene_counts_skene.get(gene, 0) + 1

    skene_predictors = {g for g, c in gene_counts_skene.items() if c >= 2}

    # ---------- Overlap ----------
    overlap = mathys_predictors & skene_predictors

    smaller_set_size = min(len(mathys_predictors), len(skene_predictors))
    recall_smaller = len(overlap) / smaller_set_size if smaller_set_size > 0 else 0.0

    print(f"Mathys predictors: {len(mathys_predictors)}")
    print(f"Skene predictors:  {len(skene_predictors)}")
    print(f"Overlap:           {len(overlap)}")
    print(f"Recall (smaller):  {recall_smaller:.3f}")

    summary_rows.append({
        "cell_type": ct,
        "n_mathys_predictors": len(mathys_predictors),
        "n_skene_predictors": len(skene_predictors),
        "n_overlap": len(overlap),
        "recall_smaller_predictor_set": recall_smaller
    })

# ============================
# Build wide-format table
# ============================
summary = pd.DataFrame(summary_rows)

wide = pd.DataFrame(index=[
    "Mathys predictors",
    "Skene predictors",
    "Overlapping predictors",
    "Recall of smaller predictor set"
])

for ct in cell_types:
    row = summary[summary["cell_type"] == ct].iloc[0]
    wide[ct] = [
        row["n_mathys_predictors"],
        row["n_skene_predictors"],
        row["n_overlap"],
        row["recall_smaller_predictor_set"],
    ]

# ============================
# Description block
# ============================
description = pd.DataFrame({
    "": [
        "Analysis: Predictor–Predictor Overlap",
        "Predictors: Mathys et al. (48) vs Skene et al.",
        "Predictor definition: nonzero importance in ≥2 of 5 splits",
        "Metric: recall of the smaller predictor set",
        "Computed independently for each cell type",
        "",
    ]
})

# ============================
# Write Excel sheet (safe logic)
# ============================
with pd.ExcelWriter(
    EXCEL_OUT,
    engine="openpyxl",
    mode="w"
) as writer:

    description.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        index=False,
        header=False,
        startrow=0
    )

    wide.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        startrow=len(description) + 1
    )

print("\n✔ Sheet written:", SHEET_NAME)
print("✔ Workbook:", EXCEL_OUT)


=== Ast ===
Mathys predictors: 175
Skene predictors:  1182
Overlap:           107
Recall (smaller):  0.611

=== Mic ===
Mathys predictors: 466
Skene predictors:  652
Overlap:           297
Recall (smaller):  0.637

=== In ===
Mathys predictors: 108
Skene predictors:  258
Overlap:           25
Recall (smaller):  0.231

=== Oli ===
Mathys predictors: 621
Skene predictors:  887
Overlap:           410
Recall (smaller):  0.660

=== Opc ===
Mathys predictors: 835
Skene predictors:  285
Overlap:           156
Recall (smaller):  0.547

=== Ex ===
Mathys predictors: 162
Skene predictors:  77
Overlap:           34
Recall (smaller):  0.442

✔ Sheet written: PredOverlap_Mathys48_vs_Skene
✔ Workbook: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/Correlation_Analyses_PredOverlap_Mathys48_vs_Skene.xlsx


In [6]:
# Mathys 48 Predictors vs Lau et al Predictors (rank vs rank)
# Standalone: loads models, computes rank correlations, writes Excel sheet

import os
import pandas as pd
import numpy as np
import joblib
from scipy.stats import spearmanr

# ============================
# PATHS
# ============================
mathys_model_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
lau_model_dir    = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Validation_Multirun_cell_on_cell_genes_Lau_rfe"

SHEET_NAME = "PredRank_Mathys48_vs_Lau"

EXCEL_OUT = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/"
    f"Correlation_Analyses_{SHEET_NAME}.xlsx"
)

os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

# Lau does NOT have OPC
cell_types = ["Ast", "Mic", "In", "Oli", "Ex"]

results = []

# ============================
# Loop per cell type
# ============================
for ct in cell_types:
    print(f"\n=== {ct} ===")

    # ---------- Load Mathys predictors ----------
    all_split_data_mathys = {}
    all_features_mathys = set()

    for split in range(1, 6):
        model_path = os.path.join(
            mathys_model_dir, ct, f"split_{split}", "maximal_classifier.joblib"
        )
        model = joblib.load(model_path)

        feats = model.feature_names_in_
        imps = model.feature_importances_

        all_features_mathys.update(feats)
        all_split_data_mathys[split] = dict(zip(feats, imps))

    rows_mathys = []
    for gene in sorted(all_features_mathys):
        norm_imps = []

        for split in range(1, 6):
            imp = all_split_data_mathys[split].get(gene, 0.0)
            split_imps = np.array(list(all_split_data_mathys[split].values()))
            denom = split_imps[split_imps > 0].sum()

            norm_imps.append(
                imp / denom if imp > 0 and denom > 0 else 0.0
            )

        rows_mathys.append({
            "gene": gene.strip(),
            "mean_importance_mathys": np.mean(norm_imps)
        })

    mathys_df = pd.DataFrame(rows_mathys)
    mathys_df["rank_mathys"] = mathys_df["mean_importance_mathys"].rank(ascending=False)

    # ---------- Load Lau predictors ----------
    all_split_data_lau = {}
    all_features_lau = set()

    for split in range(1, 6):
        model_path = os.path.join(
            lau_model_dir, ct, f"split_{split}", "maximal_classifier.joblib"
        )
        model = joblib.load(model_path)

        feats = model.feature_names_in_
        imps = model.feature_importances_

        all_features_lau.update(feats)
        all_split_data_lau[split] = dict(zip(feats, imps))

    rows_lau = []
    for gene in sorted(all_features_lau):
        norm_imps = []

        for split in range(1, 6):
            imp = all_split_data_lau[split].get(gene, 0.0)
            split_imps = np.array(list(all_split_data_lau[split].values()))
            denom = split_imps[split_imps > 0].sum()

            norm_imps.append(
                imp / denom if imp > 0 and denom > 0 else 0.0
            )

        rows_lau.append({
            "gene": gene.strip(),
            "mean_importance_lau": np.mean(norm_imps)
        })

    lau_df = pd.DataFrame(rows_lau)
    lau_df["rank_lau"] = lau_df["mean_importance_lau"].rank(ascending=False)

    # ---------- Merge + correlate ----------
    merged = mathys_df.merge(lau_df, on="gene", how="inner")

    rho, p = spearmanr(
        merged["rank_mathys"],
        merged["rank_lau"]
    )

    print(f"Spearman rho = {rho:.3f}, p = {p:.2e}, n = {merged.shape[0]}")

    results.append({
        "cell_type": ct,
        "spearman_rho": rho,
        "p_value": p,
        "n_genes": merged.shape[0]
    })

# ============================
# Build wide-format table
# ============================
summary = pd.DataFrame(results)

wide = pd.DataFrame(index=[
    "Spearman rho",
    "p-value",
    "Number of genes"
])

for ct in cell_types:
    row = summary[summary["cell_type"] == ct].iloc[0]
    wide[ct] = [
        row["spearman_rho"],
        row["p_value"],
        row["n_genes"],
    ]

# ============================
# Description block
# ============================
description = pd.DataFrame({
    "": [
        "Analysis: Predictor–Predictor Rank Correlation",
        "Predictors: Mathys et al. (48) vs Lau et al.",
        "Predictor score: Mean normalized feature importance across 5 splits",
        "Statistic: Spearman correlation of gene importance ranks",
        "Computed independently for each cell type (OPC excluded)",
        "",
    ]
})

# ============================
# Write Excel sheet
# ============================
with pd.ExcelWriter(
    EXCEL_OUT,
    engine="openpyxl",
    mode="w"
) as writer:

    description.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        index=False,
        header=False,
        startrow=0
    )

    wide.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        startrow=len(description) + 1
    )

print("\n✔ Sheet written:", SHEET_NAME)
print("✔ Workbook:", EXCEL_OUT)


=== Ast ===
Spearman rho = 0.115, p = 5.74e-10, n = 2883

=== Mic ===
Spearman rho = 0.248, p = 2.38e-21, n = 1423

=== In ===
Spearman rho = 0.039, p = 5.15e-03, n = 5196

=== Oli ===
Spearman rho = 0.368, p = 1.81e-34, n = 1034

=== Ex ===
Spearman rho = 0.192, p = 1.88e-57, n = 6819

✔ Sheet written: PredRank_Mathys48_vs_Lau
✔ Workbook: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/Correlation_Analyses_PredRank_Mathys48_vs_Lau.xlsx


In [7]:
# Mathys–Lau predictor overlap using smaller-set recall
# Predictor = nonzero importance in ≥2 splits
# Writes results to Excel sheet (standalone workbook)

import os
import joblib
import pandas as pd

# ============================
# PATHS
# ============================
mathys_model_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
lau_model_dir    = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Validation_Multirun_cell_on_cell_genes_Lau_rfe"

SHEET_NAME = "PredOverlap_Mathys48_vs_Lau"

EXCEL_OUT = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/"
    f"Correlation_Analyses_{SHEET_NAME}.xlsx"
)

os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

# Lau does NOT have OPC
cell_types = ["Ast", "Mic", "In", "Oli", "Ex"]

summary_rows = []

total_overlap = 0
total_smaller = 0

# ============================
# Loop per cell type
# ============================
for ct in cell_types:
    print(f"\n=== {ct} ===")

    # ---------- Mathys predictor set ----------
    gene_counts_mathys = {}

    for split in range(1, 6):
        model = joblib.load(
            os.path.join(
                mathys_model_dir, ct, f"split_{split}", "maximal_classifier.joblib"
            )
        )
        for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_counts_mathys[gene] = gene_counts_mathys.get(gene, 0) + 1

    mathys_predictors = {g for g, c in gene_counts_mathys.items() if c >= 2}

    # ---------- Lau predictor set ----------
    gene_counts_lau = {}

    for split in range(1, 6):
        model = joblib.load(
            os.path.join(
                lau_model_dir, ct, f"split_{split}", "maximal_classifier.joblib"
            )
        )
        for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_counts_lau[gene] = gene_counts_lau.get(gene, 0) + 1

    lau_predictors = {g for g, c in gene_counts_lau.items() if c >= 2}

    # ---------- Overlap ----------
    overlap = mathys_predictors & lau_predictors

    smaller_set_size = min(len(mathys_predictors), len(lau_predictors))
    recall_smaller = len(overlap) / smaller_set_size if smaller_set_size > 0 else 0

    print(f"Mathys predictors: {len(mathys_predictors)}")
    print(f"Lau predictors:    {len(lau_predictors)}")
    print(f"Overlap:           {len(overlap)}")
    print(f"Recall (smaller set): {recall_smaller:.3f}")

    summary_rows.append({
        "cell_type": ct,
        "n_mathys_predictors": len(mathys_predictors),
        "n_lau_predictors": len(lau_predictors),
        "n_overlap": len(overlap),
        "recall_smaller_set": recall_smaller
    })

    total_overlap += len(overlap)
    total_smaller += smaller_set_size

# ============================
# Global recall across cell types
# ============================
global_recall = total_overlap / total_smaller if total_smaller > 0 else 0
print(f"\nGLOBAL predictor recall (smaller set): {global_recall:.3f}")

summary_df = pd.DataFrame(summary_rows)
summary_df.loc[len(summary_df)] = {
    "cell_type": "GLOBAL",
    "n_mathys_predictors": "",
    "n_lau_predictors": "",
    "n_overlap": total_overlap,
    "recall_smaller_set": global_recall
}

# ============================
# Build wide-format table
# ============================
wide = pd.DataFrame(index=[
    "Mathys predictors",
    "Lau predictors",
    "Overlapping predictors",
    "Recall of smaller predictor set"
])

for ct in cell_types:
    row = summary_df[summary_df["cell_type"] == ct].iloc[0]
    wide[ct] = [
        row["n_mathys_predictors"],
        row["n_lau_predictors"],
        row["n_overlap"],
        row["recall_smaller_set"],
    ]

# Add GLOBAL column
wide["GLOBAL"] = [
    "",
    "",
    total_overlap,
    global_recall
]

# ============================
# Description block
# ============================
description = pd.DataFrame({
    "": [
        "Analysis: Predictor–Predictor Overlap (Smaller-Set Recall)",
        "Predictors: Mathys et al. (48) vs Lau et al.",
        "Predictor definition: nonzero importance in ≥2 of 5 splits",
        "Metric: recall of the smaller predictor set",
        "Includes global recall aggregated across cell types",
        "Computed independently per cell type (OPC excluded)",
        "",
    ]
})

# ============================
# Write Excel sheet
# ============================
with pd.ExcelWriter(
    EXCEL_OUT,
    engine="openpyxl",
    mode="w"
) as writer:

    description.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        index=False,
        header=False,
        startrow=0
    )

    wide.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        startrow=len(description) + 1
    )

print("\n✔ Sheet written:", SHEET_NAME)
print("✔ Workbook:", EXCEL_OUT)


=== Ast ===
Mathys predictors: 175
Lau predictors:    212
Overlap:           40
Recall (smaller set): 0.229

=== Mic ===
Mathys predictors: 466
Lau predictors:    274
Overlap:           122
Recall (smaller set): 0.445

=== In ===
Mathys predictors: 108
Lau predictors:    50
Overlap:           12
Recall (smaller set): 0.240

=== Oli ===
Mathys predictors: 621
Lau predictors:    268
Overlap:           106
Recall (smaller set): 0.396

=== Ex ===
Mathys predictors: 162
Lau predictors:    17
Overlap:           14
Recall (smaller set): 0.824

GLOBAL predictor recall (smaller set): 0.375

✔ Sheet written: PredOverlap_Mathys48_vs_Lau
✔ Workbook: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/Correlation_Analyses_PredOverlap_Mathys48_vs_Lau.xlsx


In [8]:
# Random-effect DEG vs Fixed-effect DEG (Mathys 48, 2019)
# Spearman correlation on raw log2FC
# Writes results to Excel sheet (standalone workbook)

import os
import pandas as pd
from scipy.stats import spearmanr

# ============================
# PATHS
# ============================
deg_re_dir    = "/n/scratch/users/a/adm808/Revision/Clincal_batch_DE_Outputs_revision"
deg_fixed_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/Differential_Expression_Final/Fixed"

SHEET_NAME = "DEG_RE_vs_Fixed_Mathys48_log2FC"

EXCEL_OUT = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/"
    f"Correlation_Analyses_{SHEET_NAME}.xlsx"
)

os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

results = []

# ============================
# Loop per cell type
# ============================
for ct in cell_types:
    print(f"\n=== {ct} ===")

    re_path = os.path.join(deg_re_dir,    f"poisson_DE_results_{ct}.csv")
    fx_path = os.path.join(deg_fixed_dir, f"poisson_DE_results_{ct}.csv")

    deg_re = pd.read_csv(re_path)
    deg_fx = pd.read_csv(fx_path)

    # Clean gene names
    deg_re["gene"] = deg_re["gene"].str.strip()
    deg_fx["gene"] = deg_fx["gene"].str.strip()

    # Drop missing log2FC
    deg_re = deg_re.dropna(subset=["log2FC"])
    deg_fx = deg_fx.dropna(subset=["log2FC"])

    # Merge on gene
    merged = deg_re[["gene", "log2FC"]].merge(
        deg_fx[["gene", "log2FC"]],
        on="gene",
        suffixes=("_random_effect", "_fixed_effect"),
        how="inner"
    )

    # Spearman correlation on log2FC
    rho, p = spearmanr(
        merged["log2FC_random_effect"],
        merged["log2FC_fixed_effect"]
    )

    print(f"Spearman rho = {rho:.3f}, p = {p:.2e}, n = {merged.shape[0]}")

    results.append({
        "cell_type": ct,
        "spearman_rho": rho,
        "p_value": p,
        "n_genes": merged.shape[0]
    })

# ============================
# Build wide-format table
# ============================
summary = pd.DataFrame(results)

wide = pd.DataFrame(index=[
    "Spearman rho",
    "p-value",
    "Number of genes"
])

for ct in cell_types:
    row = summary[summary["cell_type"] == ct].iloc[0]
    wide[ct] = [
        row["spearman_rho"],
        row["p_value"],
        row["n_genes"],
    ]

# ============================
# Description block
# ============================
description = pd.DataFrame({
    "": [
        "Analysis: Random-effect vs Fixed-effect DEG Correlation",
        "Dataset: Mathys et al. (2019)",
        "Random-effect model vs fixed-effect model (age, sex, PMI only)",
        "Statistic: Spearman correlation of raw log2 fold change",
        "Computed independently for each cell type",
        "",
    ]
})

# ============================
# Write Excel sheet
# ============================
with pd.ExcelWriter(
    EXCEL_OUT,
    engine="openpyxl",
    mode="w"
) as writer:

    description.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        index=False,
        header=False,
        startrow=0
    )

    wide.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        startrow=len(description) + 1
    )

print("\n✔ Sheet written:", SHEET_NAME)
print("✔ Workbook:", EXCEL_OUT)


=== Ast ===
Spearman rho = 0.867, p = 0.00e+00, n = 13732

=== Mic ===
Spearman rho = 0.767, p = 0.00e+00, n = 10807

=== In ===
Spearman rho = 0.826, p = 0.00e+00, n = 15603

=== Oli ===
Spearman rho = 0.870, p = 0.00e+00, n = 14850

=== Opc ===
Spearman rho = 0.842, p = 0.00e+00, n = 13433

=== Ex ===
Spearman rho = 0.856, p = 0.00e+00, n = 17017

✔ Sheet written: DEG_RE_vs_Fixed_Mathys48_log2FC
✔ Workbook: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/Correlation_Analyses_DEG_RE_vs_Fixed_Mathys48_log2FC.xlsx


In [9]:
# Random-effect DEG vs Fixed-effect DEG with age, sex, PMI, batch, APOE (Mathys 48)
# Spearman correlation on raw log2FC
# Writes results to Excel sheet (standalone workbook)

import os
import pandas as pd
from scipy.stats import spearmanr

# ============================
# PATHS
# ============================
deg_re_dir    = "/n/scratch/users/a/adm808/Revision/Clincal_batch_DE_Outputs_revision"
deg_fixed_dir = "/n/scratch/users/a/adm808/Revision/Fixed_effects_batch_corrected/Clincal_batch_DE_Outputs"

SHEET_NAME = "DEG_RE_vs_FixedBatchApoe_Mathys48_log2FC"

EXCEL_OUT = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/"
    f"Correlation_Analyses_{SHEET_NAME}.xlsx"
)

os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

results = []

# ============================
# Loop per cell type
# ============================
for ct in cell_types:
    print(f"\n=== {ct} ===")

    re_path = os.path.join(deg_re_dir,    f"poisson_DE_results_{ct}.csv")
    fx_path = os.path.join(deg_fixed_dir, f"poisson_DE_results_{ct}.csv")

    deg_re = pd.read_csv(re_path)
    deg_fx = pd.read_csv(fx_path)

    # Clean gene names
    deg_re["gene"] = deg_re["gene"].str.strip()
    deg_fx["gene"] = deg_fx["gene"].str.strip()

    # Drop missing log2FC
    deg_re = deg_re.dropna(subset=["log2FC"])
    deg_fx = deg_fx.dropna(subset=["log2FC"])

    # Merge on gene
    merged = deg_re[["gene", "log2FC"]].merge(
        deg_fx[["gene", "log2FC"]],
        on="gene",
        suffixes=("_random_effect", "_fixed_effect"),
        how="inner"
    )

    # Spearman correlation on log2FC
    rho, p = spearmanr(
        merged["log2FC_random_effect"],
        merged["log2FC_fixed_effect"]
    )

    print(f"Spearman rho = {rho:.3f}, p = {p:.2e}, n = {merged.shape[0]}")

    results.append({
        "cell_type": ct,
        "spearman_rho": rho,
        "p_value": p,
        "n_genes": merged.shape[0]
    })

# ============================
# Build wide-format table
# ============================
summary = pd.DataFrame(results)

wide = pd.DataFrame(index=[
    "Spearman rho",
    "p-value",
    "Number of genes"
])

for ct in cell_types:
    row = summary[summary["cell_type"] == ct].iloc[0]
    wide[ct] = [
        row["spearman_rho"],
        row["p_value"],
        row["n_genes"],
    ]

# ============================
# Description block
# ============================
description = pd.DataFrame({
    "": [
        "Analysis: Random-effect vs Fixed-effect DEG Correlation",
        "Dataset: Mathys et al. (2019)",
        "Random-effect model vs fixed-effect model (age, sex, PMI, batch, APOE genotype)",
        "Statistic: Spearman correlation of raw log2 fold change",
        "Computed independently for each cell type",
        "",
    ]
})

# ============================
# Write Excel sheet
# ============================
with pd.ExcelWriter(
    EXCEL_OUT,
    engine="openpyxl",
    mode="w"
) as writer:

    description.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        index=False,
        header=False,
        startrow=0
    )

    wide.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        startrow=len(description) + 1
    )

print("\n✔ Sheet written:", SHEET_NAME)
print("✔ Workbook:", EXCEL_OUT)


=== Ast ===
Spearman rho = 0.716, p = 0.00e+00, n = 13732

=== Mic ===
Spearman rho = 0.629, p = 0.00e+00, n = 10807

=== In ===
Spearman rho = 0.675, p = 0.00e+00, n = 15603

=== Oli ===
Spearman rho = 0.705, p = 0.00e+00, n = 14850

=== Opc ===
Spearman rho = 0.703, p = 0.00e+00, n = 13433

=== Ex ===
Spearman rho = 0.777, p = 0.00e+00, n = 17017

✔ Sheet written: DEG_RE_vs_FixedBatchApoe_Mathys48_log2FC
✔ Workbook: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/Correlation_Analyses_DEG_RE_vs_FixedBatchApoe_Mathys48_log2FC.xlsx


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


In [10]:
# Predictor vs DEG (Mathys 2024) — overlap % + Fisher OR + FDR across cell types
# Excel output only (no CSVs, no prints beyond loop progress)

import os
import numpy as np
import pandas as pd
import joblib
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

# ============================
# Paths
# ============================
deg423_dir  = "/n/scratch/users/a/adm808/Revision/New_Data_DEG_results"
model_dir   = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"

SHEET_NAME = "Pred_vs_DEG_Mathys2024_Fisher"

EXCEL_OUT = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/"
    f"Correlation_Analyses_{SHEET_NAME}.xlsx"
)
os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

# Predictor definition
MIN_SPLITS = 2

# DEG definition
P_ADJ_CUTOFF = 0.05
ABS_LOG2FC_CUTOFF = 0.25

rows = []

# ============================
# Loop per cell type
# ============================
for ct in cell_types:
    print(f"\n=== {ct} ===")

    # ---------- Load DEG (Mathys 2024) ----------
    deg_path = os.path.join(deg423_dir, f"poisson_DE_results_PFC_{ct}.csv")
    deg = pd.read_csv(deg_path)

    deg["gene"] = deg["gene"].astype(str).str.strip()
    deg["p_adj"] = pd.to_numeric(deg["p_adj"], errors="coerce")
    deg["log2FC"] = pd.to_numeric(deg["log2FC"], errors="coerce")

    # DEG-tested universe
    deg_tested = deg.dropna(subset=["p_adj", "log2FC"])
    tested_genes = set(deg_tested["gene"])

    # Significant DEGs
    deg_sig = deg_tested[
        (deg_tested["p_adj"] < P_ADJ_CUTOFF) &
        (deg_tested["log2FC"].abs() > ABS_LOG2FC_CUTOFF)
    ]
    deg_sig_genes = set(deg_sig["gene"])

    # ---------- Load predictors ----------
    gene_counts = {}
    for split in range(1, 6):
        model = joblib.load(
            os.path.join(model_dir, ct, f"split_{split}", "maximal_classifier.joblib")
        )
        for g, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                g = str(g).strip()
                gene_counts[g] = gene_counts.get(g, 0) + 1

    predictors = {g for g, c in gene_counts.items() if c >= MIN_SPLITS}

    # Restrict predictors to DEG-tested universe
    predictors_tested = predictors & tested_genes

    # Overlap
    overlap = predictors_tested & deg_sig_genes
    overlap_pct = (
        len(overlap) / len(predictors_tested)
        if len(predictors_tested) > 0 else np.nan
    )

    # ---------- Fisher enrichment ----------
    a = len(overlap)
    b = len(predictors_tested) - a
    c = len(deg_sig_genes - predictors_tested)
    d = len(tested_genes) - (a + b + c)

    if d < 0:
        raise ValueError(f"Negative d for {ct}. Check universe definitions.")

    OR, p = fisher_exact([[a, b], [c, d]], alternative="two-sided")

    rows.append({
        "cell_type": ct,
        "n_tested_genes": len(tested_genes),
        "n_predictors_total": len(predictors),
        "n_predictors_tested": len(predictors_tested),
        "n_deg_sig": len(deg_sig_genes),
        "n_overlap": a,
        "overlap_pct_of_predictors_tested": overlap_pct,
        "odds_ratio": OR,
        "p_value": p
    })

# ============================
# Summary + FDR
# ============================
summary = pd.DataFrame(rows)
summary["p_value_fdr"] = multipletests(
    summary["p_value"], method="fdr_bh"
)[1]

# ============================
# Wide-format table
# ============================
wide = pd.DataFrame(index=[
    "Tested genes (DEG universe)",
    "Predictors (total)",
    "Predictors (tested universe)",
    "Significant DEGs",
    "Overlap",
    "Overlap % of predictors (tested)",
    "Odds ratio (Fisher)",
    "p-value",
    "FDR-adjusted p-value"
])

for ct in cell_types:
    row = summary[summary["cell_type"] == ct].iloc[0]
    wide[ct] = [
        row["n_tested_genes"],
        row["n_predictors_total"],
        row["n_predictors_tested"],
        row["n_deg_sig"],
        row["n_overlap"],
        row["overlap_pct_of_predictors_tested"],
        row["odds_ratio"],
        row["p_value"],
        row["p_value_fdr"],
    ]

# ============================
# Description block
# ============================
description = pd.DataFrame({
    "": [
        "Analysis: Predictor vs DEG Overlap Enrichment (Fisher exact test)",
        "DEG source: Mathys et al. (2024), PFC Poisson differential expression",
        "Predictors: stable predictors (Mathys 48 - 2019)",
        "Universe: genes tested in DEG table (non-missing p_adj and log2FC)",
        "DEG definition: p_adj < 0.05 and |log2FC| > 0.25",
        "Statistic: Fisher exact test (two-sided); FDR (BH) across cell types",
        "",
    ]
})

# ============================
# Write Excel sheet ONLY
# ============================
with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl", mode="w") as writer:
    description.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        index=False,
        header=False,
        startrow=0
    )
    wide.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        startrow=len(description) + 1
    )

print("\n✔ Sheet written:", SHEET_NAME)
print("✔ Workbook:", EXCEL_OUT)


=== Ast ===

=== Mic ===

=== In ===

=== Oli ===

=== Opc ===

=== Ex ===

✔ Sheet written: Pred_vs_DEG_Mathys2024_Fisher
✔ Workbook: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/Correlation_Analyses_Pred_vs_DEG_Mathys2024_Fisher.xlsx


In [11]:
# # Predictor vs DEG (Mathys 2024) — Spearman between mean predictor importance and abs(log2FC)
# # Excel output only; prints preserved

# import os
# import numpy as np
# import pandas as pd
# import joblib
# from scipy.stats import spearmanr
# from statsmodels.stats.multitest import multipletests

# # ============================
# # Paths
# # ============================
# deg423_dir  = "/n/scratch/users/a/adm808/Revision/New_Data_DEG_results"
# model_dir   = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"

# SHEET_NAME = "PredImp_vs_abslog2FC_Mathys2024"

# EXCEL_OUT = (
#     "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/"
#     f"Correlation_Analyses_{SHEET_NAME}.xlsx"
# )
# os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

# cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

# MIN_SPLITS = 2

# rows = []

# # ============================
# # Loop per cell type
# # ============================
# for ct in cell_types:
#     print(f"\n=== {ct} ===")

#     # ---------- Load DEG (Mathys 2024) ----------
#     deg_path = os.path.join(deg423_dir, f"poisson_DE_results_PFC_{ct}.csv")
#     deg = pd.read_csv(deg_path)

#     deg["gene"] = deg["gene"].astype(str).str.strip()
#     deg["log2FC"] = pd.to_numeric(deg["log2FC"], errors="coerce")
#     deg = deg.dropna(subset=["log2FC"])

#     tested_genes = set(deg["gene"])
#     deg_absfc = deg[["gene", "log2FC"]].copy()
#     deg_absfc["abs_log2FC"] = deg_absfc["log2FC"].abs()

#     # ---------- Load model importances ----------
#     split_imps = {}
#     all_genes = set()

#     for split in range(1, 6):
#         model = joblib.load(
#             os.path.join(model_dir, ct, f"split_{split}", "maximal_classifier.joblib")
#         )
#         split_imps[split] = dict(zip(model.feature_names_in_, model.feature_importances_))
#         all_genes.update(model.feature_names_in_)

#     rows_imp = []
#     for g in all_genes:
#         g = str(g).strip()
#         nonzero_splits = 0
#         norm_imps = []

#         for s in split_imps:
#             imp = split_imps[s].get(g, 0.0)
#             denom = sum(v for v in split_imps[s].values() if v > 0)

#             if imp > 0:
#                 nonzero_splits += 1

#             norm_imps.append(imp / denom if imp > 0 and denom > 0 else 0.0)

#         if nonzero_splits >= MIN_SPLITS:
#             rows_imp.append({
#                 "gene": g,
#                 "mean_importance": float(np.mean(norm_imps)),
#                 "n_splits_nonzero": nonzero_splits
#             })

#     imp_df = pd.DataFrame(rows_imp)

#     # Restrict to DEG-tested genes
#     imp_df = imp_df[imp_df["gene"].isin(tested_genes)].copy()

#     merged = imp_df.merge(
#         deg_absfc[["gene", "abs_log2FC"]],
#         on="gene",
#         how="inner"
#     )

#     if merged.shape[0] < 10:
#         rho, p = np.nan, np.nan
#     else:
#         rho, p = spearmanr(
#             merged["mean_importance"],
#             merged["abs_log2FC"]
#         )

#     print(
#         f"n_predictors_tested={merged.shape[0]} | "
#         f"Spearman rho={rho:.3f} | raw p={p:.2e}"
#     )

#     rows.append({
#         "cell_type": ct,
#         "spearman_rho": rho,
#         "p_value": p,
#         "n_predictors_tested": merged.shape[0]
#     })

# # ============================
# # Summary + FDR
# # ============================
# summary = pd.DataFrame(rows)
# summary["p_value_fdr"] = multipletests(
#     summary["p_value"], method="fdr_bh"
# )[1]

# print("\n=== FDR-adjusted Predictor importance vs abs(log2FC) (Mathys 2024) Spearman ===")
# print(
#     summary[
#         ["cell_type", "spearman_rho", "p_value", "p_value_fdr", "n_predictors_tested"]
#     ]
# )

# # ============================
# # Build wide-format table
# # ============================
# wide = pd.DataFrame(index=[
#     "Spearman rho",
#     "p-value",
#     "FDR-adjusted p-value",
#     "Number of predictors tested"
# ])

# for ct in cell_types:
#     row = summary[summary["cell_type"] == ct].iloc[0]
#     wide[ct] = [
#         row["spearman_rho"],
#         row["p_value"],
#         row["p_value_fdr"],
#         row["n_predictors_tested"],
#     ]

# # ============================
# # Description block
# # ============================
# description = pd.DataFrame({
#     "": [
#         "Analysis: Predictor importance vs DEG effect size correlation",
#         "DEG source: Mathys et al. (2024), PFC Poisson differential expression",
#         "Predictors: stable predictors (nonzero importance in ≥2 of 5 splits)",
#         "Predictor score: mean normalized feature importance",
#         "Statistic: Spearman correlation with |log2FC|; FDR (BH) across cell types",
#         "",
#     ]
# })

# # ============================
# # Write Excel sheet ONLY
# # ============================
# with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl", mode="w") as writer:
#     description.to_excel(
#         writer,
#         sheet_name=SHEET_NAME,
#         index=False,
#         header=False,
#         startrow=0
#     )
#     wide.to_excel(
#         writer,
#         sheet_name=SHEET_NAME,
#         startrow=len(description) + 1
#     )

# print("\n✔ Sheet written:", SHEET_NAME)
# print("✔ Workbook:", EXCEL_OUT)

In [12]:
# Predictor vs Random Effect DEG for Mathys 2024
# Rank–rank Spearman: predictor rank vs |log2FC| rank
# LOGIC IDENTICAL TO Mathys 427 (only DEG source differs)

import os
import pandas as pd
import numpy as np
import joblib
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# ============================
# Paths
# ============================
deg_dir = "/n/scratch/users/a/adm808/Revision/New_Data_DEG_results"
model_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"

SHEET_NAME = "PredRank_vs_abslog2FC_REDEG_Mathys2024"

EXCEL_OUT = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/"
    f"Correlation_Analyses_{SHEET_NAME}.xlsx"
)
os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

rows = []

# ============================
# Loop per cell type
# ============================
for ct in cell_types:
    print(f"\n=== {ct} ===")

    # ---------- Load DEG ----------
    deg_path = os.path.join(deg_dir, f"poisson_DE_results_PFC_{ct}.csv")
    deg = pd.read_csv(deg_path)

    deg["gene"] = deg["gene"].astype(str).str.strip()
    deg["log2FC"] = pd.to_numeric(deg["log2FC"], errors="coerce")
    deg = deg.dropna(subset=["log2FC"])

    # Rank by absolute log2FC (AVERAGE tie handling)
    deg["abs_log2FC"] = deg["log2FC"].abs()
    deg["deg_rank"] = deg["abs_log2FC"].rank(
        ascending=False,
        method="average"
    )

    tested_genes = set(deg["gene"])

    # ---------- Load predictors ----------
    split_imps = {}
    all_genes = set()

    for split in range(1, 6):
        model = joblib.load(
            os.path.join(model_dir, ct, f"split_{split}", "maximal_classifier.joblib")
        )
        split_imps[split] = dict(
            zip(model.feature_names_in_, model.feature_importances_)
        )
        all_genes.update(model.feature_names_in_)

    rows_imp = []
    for g in all_genes:
        g = str(g).strip()
        norm_imps = []

        for s in split_imps:
            imp = split_imps[s].get(g, 0.0)
            denom = sum(v for v in split_imps[s].values() if v > 0)
            norm_imps.append(imp / denom if imp > 0 and denom > 0 else 0.0)

        rows_imp.append({
            "gene": g,
            "mean_importance": np.mean(norm_imps)
        })

    imp_df = pd.DataFrame(rows_imp)
    imp_df = imp_df[imp_df["gene"].isin(tested_genes)].copy()

    # Rank predictors (AVERAGE tie handling)
    imp_df["pred_rank"] = imp_df["mean_importance"].rank(
        ascending=False,
        method="average"
    )

    # ---------- Merge and correlate ----------
    merged = deg[["gene", "deg_rank"]].merge(
        imp_df[["gene", "pred_rank"]],
        on="gene",
        how="inner"
    )

    if merged.shape[0] < 10:
        rho, p = np.nan, np.nan
    else:
        rho, p = spearmanr(
            merged["deg_rank"],
            merged["pred_rank"]
        )

    print(
        f"n_genes={merged.shape[0]} | "
        f"Spearman rho={rho:.3f} | raw p={p:.2e}"
    )

    rows.append({
        "cell_type": ct,
        "spearman_rho": rho,
        "p_value": p,
        "n_genes": merged.shape[0]
    })

# ============================
# Summary + FDR
# ============================
summary = pd.DataFrame(rows)
summary["p_value_fdr"] = multipletests(
    summary["p_value"], method="fdr_bh"
)[1]

print("\n=== FDR-adjusted Rank–Rank Predictor vs |log2FC| correlations (Mathys 2024) ===")
print(
    summary[["cell_type", "spearman_rho", "p_value", "p_value_fdr", "n_genes"]]
)

# ============================
# Wide-format table
# ============================
wide = pd.DataFrame(index=[
    "Spearman rho",
    "p-value",
    "FDR-adjusted p-value",
    "Number of genes"
])

for ct in cell_types:
    row = summary[summary["cell_type"] == ct].iloc[0]
    wide[ct] = [
        row["spearman_rho"],
        row["p_value"],
        row["p_value_fdr"],
        row["n_genes"],
    ]

# ============================
# Description block
# ============================
description = pd.DataFrame({
    "": [
        "Analysis: Predictor vs random-effects DEG rank concordance",
        "Dataset: Mathys et al. (2024), PFC",
        "DEG model: Poisson differential expression",
        "DEG ranking: absolute log2 fold change (all genes tested)",
        "Predictor ranking: mean normalized feature importance",
        "Statistic: Spearman rank correlation (rank–rank)",
        "Multiple testing: BH-FDR across cell types",
        "",
    ]
})

# ============================
# Write Excel ONLY
# ============================
with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl", mode="w") as writer:
    description.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        index=False,
        header=False,
        startrow=0
    )
    wide.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        startrow=len(description) + 1
    )

print("\n✔ Sheet written:", SHEET_NAME)
print("✔ Workbook:", EXCEL_OUT)


=== Ast ===
n_genes=2793 | Spearman rho=0.045 | raw p=1.85e-02

=== Mic ===
n_genes=1359 | Spearman rho=0.170 | raw p=2.84e-10

=== In ===
n_genes=5198 | Spearman rho=0.023 | raw p=1.02e-01

=== Oli ===
n_genes=1001 | Spearman rho=0.125 | raw p=6.92e-05

=== Opc ===
n_genes=3601 | Spearman rho=0.009 | raw p=5.82e-01

=== Ex ===
n_genes=7157 | Spearman rho=0.095 | raw p=6.33e-16

=== FDR-adjusted Rank–Rank Predictor vs |log2FC| correlations (Mathys 2024) ===
  cell_type  spearman_rho       p_value   p_value_fdr  n_genes
0       Ast      0.044552  1.854069e-02  2.781103e-02     2793
1       Mic      0.169998  2.843074e-10  8.529222e-10     1359
2        In      0.022683  1.020133e-01  1.224159e-01     5198
3       Oli      0.125426  6.918535e-05  1.383707e-04     1001
4       Opc      0.009183  5.817231e-01  5.817231e-01     3601
5        Ex      0.095342  6.327204e-16  3.796323e-15     7157

✔ Sheet written: PredRank_vs_abslog2FC_REDEG_Mathys2024
✔ Workbook: /n/groups/patel/adithya/Alz

/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


In [13]:
# # External DEG vs Predictor validation — Spearman on abs(log2FC)
# # Excel output only; prints preserved

# import os
# import pandas as pd
# import numpy as np
# import joblib
# from scipy.stats import spearmanr

# # ============================
# # Paths
# # ============================
# external_dir = "/n/scratch/users/a/adm808/Revision/External_DEGs"
# model_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"

# SHEET_NAME = "ExternalDEG_vs_Predictor_abslog2FC"

# EXCEL_OUT = (
#     "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/"
#     f"Correlation_Analyses_{SHEET_NAME}.xlsx"
# )
# os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

# cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

# # ============================
# # Load predictors ONCE
# # Definition: non-zero importance in ≥2 splits
# # ============================
# predictor_tables = {}

# MIN_SPLITS = 2

# for ct in cell_types:
#     split_imps = {}
#     genes = set()

#     for split in range(1, 6):
#         model = joblib.load(
#             os.path.join(model_dir, ct, f"split_{split}", "maximal_classifier.joblib")
#         )
#         split_imps[split] = dict(
#             zip(model.feature_names_in_, model.feature_importances_)
#         )
#         genes.update(model.feature_names_in_)

#     rows = []
#     for g in genes:
#         norm = []
#         nonzero_splits = 0

#         for s in split_imps:
#             imp = split_imps[s].get(g, 0.0)
#             denom = sum(v for v in split_imps[s].values() if v > 0)

#             if imp > 0:
#                 nonzero_splits += 1

#             norm.append(imp / denom if imp > 0 and denom > 0 else 0.0)

#         if nonzero_splits >= MIN_SPLITS:
#             rows.append({
#                 "gene": g,
#                 "mean_importance": np.mean(norm),
#                 "n_splits_nonzero": nonzero_splits
#             })

#     df = pd.DataFrame(rows)
#     df["pred_rank"] = df["mean_importance"].rank(ascending=False)
#     predictor_tables[ct] = df[["gene", "pred_rank", "n_splits_nonzero"]]

# # ============================
# # Helper
# # ============================
# def corr_abs_fc(deg, ct, gene_col, fc_col):
#     deg = deg.copy()
#     deg[gene_col] = deg[gene_col].astype(str).str.strip()
#     deg[fc_col] = pd.to_numeric(deg[fc_col], errors="coerce")
#     deg = deg.dropna(subset=[fc_col])

#     deg["abs_fc"] = deg[fc_col].abs()
#     deg = deg.sort_values("abs_fc", ascending=False)
#     deg["deg_rank"] = np.arange(1, len(deg) + 1)

#     merged = deg.merge(
#         predictor_tables[ct],
#         left_on=gene_col,
#         right_on="gene",
#         how="inner"
#     )

#     if merged.shape[0] < 10:
#         return np.nan

#     return spearmanr(merged["deg_rank"], merged["pred_rank"])[0]

# # ============================
# # Collect correlations
# # ============================
# records = []

# # ---- Morabito ----
# morabito = pd.read_excel(
#     os.path.join(external_dir, "Morabito_DEG_results.xlsx"),
#     sheet_name="Supplementary Data 1c",
#     skiprows=2
# )

# mor_map = {
#     "ASC": "Ast", "MG": "Mic", "INH": "In",
#     "ODC": "Oli", "OPC": "Opc", "EX": "Ex"
# }

# for k, ct in mor_map.items():
#     r = corr_abs_fc(morabito[morabito["cluster"] == k], ct, "gene", "avg_logFC")
#     if not np.isnan(r):
#         records.append(("Morabito", ct, r))

# # ---- Sadick ----
# for ct, fname, up, down in [
#     ("Ast", "Sadick_Ast_DEG_results.xlsx",
#      "Astro_upregulated_DEGs_dis", "Astro_downregulated_DEGs_dis"),
#     ("Oli", "Sadick_Oli_DEG_results.xlsx",
#      "Oligo_upregulated_DEGs_dis", "Oligo_downregulated_DEGs_dis")
# ]:
#     dfs = []
#     for sh in [up, down]:
#         df = pd.read_excel(os.path.join(external_dir, fname), sheet_name=sh)
#         gcol = [c for c in df.columns if "gene" in c.lower()][0]
#         dfs.append(df[[gcol, "log2FC"]].rename(columns={gcol: "gene"}))
#     pooled = pd.concat(dfs).drop_duplicates("gene")

#     r = corr_abs_fc(pooled, ct, "gene", "log2FC")
#     if not np.isnan(r):
#         records.append(("Sadick", ct, r))

# # ---- Su ----
# su = pd.read_excel(
#     os.path.join(external_dir, "Su_DEG_results.xlsx"),
#     sheet_name="TableS5A",
#     skiprows=1
# )

# su_map = {
#     "AST": "Ast", "MG": "Mic",
#     "ODC": "Oli", "OLIG": "Oli",
#     "OPC": "Opc", "Glut.N": "Ex",
#     "GABA.N": "In"
# }

# for k, ct in su_map.items():
#     r = corr_abs_fc(su[su["Cluster ID"] == k], ct, "Gene", "avg_logFC")
#     if not np.isnan(r):
#         records.append(("Su", ct, r))

# # ---- Grubman ----
# grub = pd.read_excel(
#     os.path.join(external_dir, "Grubman_DEG_results.xlsx"),
#     sheet_name="Supplementary Table 2",
#     header=5
# )

# grub_map = {
#     "mg": "Mic", "astro": "Ast",
#     "oligo": "Oli", "OPC": "Opc"
# }

# for k, ct in grub_map.items():
#     r = corr_abs_fc(grub[grub["group"] == k], ct, "geneID", "logFC")
#     if not np.isnan(r):
#         records.append(("Grubman", ct, r))

# # ============================
# # Final summaries
# # ============================
# df = pd.DataFrame(records, columns=["dataset", "cell_type", "rho"])

# celltype_summary = (
#     df.groupby("cell_type")
#       .agg(mean_spearman_rho=("rho", "mean"),
#            n_datasets=("rho", "count"))
#       .reset_index()
# )

# print(celltype_summary)
# print("\nEXTERNAL DEG VALIDATION.")

# # ============================
# # Write Excel ONLY
# # ============================
# description = pd.DataFrame({
#     "": [
#         "Analysis: External DEG vs Predictors (Mathys 48 (2019))",
#         "Statistic: Spearman correlation between predictor rank and |log2FC| rank",
#         "Predictors: stable predictors (nonzero importance in ≥2 of 5 splits)",
#         "DEGs: multiple external studies (Morabito, Sadick, Su, Grubman)",
#         "DEG ranking: absolute log2 fold change",
#         "",
#     ]
# })

# with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl", mode="w") as writer:
#     description.to_excel(
#         writer,
#         sheet_name=SHEET_NAME,
#         index=False,
#         header=False,
#         startrow=0
#     )
#     celltype_summary.to_excel(
#         writer,
#         sheet_name=SHEET_NAME,
#         startrow=len(description) + 1,
#         index=False
#     )
#     df.to_excel(
#         writer,
#         sheet_name=SHEET_NAME,
#         startrow=len(description) + len(celltype_summary) + 4,
#         index=False
#     )

# print("\n✔ Sheet written:", SHEET_NAME)
# print("✔ Workbook:", EXCEL_OUT)

In [14]:
# # External DEG vs Predictor validation — Spearman on abs(log2FC)
# # Excel output only; prints preserved

# import os
# import pandas as pd
# import numpy as np
# import joblib
# from scipy.stats import spearmanr
# from statsmodels.stats.multitest import multipletests

# # ============================
# # Paths
# # ============================
# external_dir = "/n/scratch/users/a/adm808/Revision/External_DEGs"
# model_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"

# SHEET_NAME = "ExternalDEG_vs_Predictor_abslog2FC"

# EXCEL_OUT = (
#     "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/"
#     f"Correlation_Analyses_{SHEET_NAME}.xlsx"
# )
# os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

# cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

# # ============================
# # Load predictors ONCE
# # ============================
# predictor_tables = {}
# MIN_SPLITS = 2

# for ct in cell_types:
#     split_imps = {}
#     genes = set()

#     for split in range(1, 6):
#         model = joblib.load(
#             os.path.join(model_dir, ct, f"split_{split}", "maximal_classifier.joblib")
#         )
#         split_imps[split] = dict(
#             zip(model.feature_names_in_, model.feature_importances_)
#         )
#         genes.update(model.feature_names_in_)

#     rows = []
#     for g in genes:
#         norm = []
#         nonzero_splits = 0

#         for s in split_imps:
#             imp = split_imps[s].get(g, 0.0)
#             denom = sum(v for v in split_imps[s].values() if v > 0)

#             if imp > 0:
#                 nonzero_splits += 1
#             norm.append(imp / denom if imp > 0 and denom > 0 else 0.0)

#         if nonzero_splits >= MIN_SPLITS:
#             rows.append({
#                 "gene": g,
#                 "mean_importance": np.mean(norm),
#                 "n_splits_nonzero": nonzero_splits
#             })

#     df = pd.DataFrame(rows)
#     df["pred_rank"] = df["mean_importance"].rank(ascending=False)
#     predictor_tables[ct] = df[["gene", "pred_rank", "n_splits_nonzero"]]

# # ============================
# # Helper — ASYMPTOTIC Spearman (same as Mathys427)
# # ============================
# def corr_abs_fc_spearman(deg, ct, gene_col, fc_col):
#     deg = deg.copy()
#     deg[gene_col] = deg[gene_col].astype(str).str.strip()
#     deg[fc_col] = pd.to_numeric(deg[fc_col], errors="coerce")
#     deg = deg.dropna(subset=[fc_col])

#     deg["abs_fc"] = deg[fc_col].abs()
#     deg = deg.sort_values("abs_fc", ascending=False)
#     deg["deg_rank"] = np.arange(1, len(deg) + 1)

#     merged = deg.merge(
#         predictor_tables[ct],
#         left_on=gene_col,
#         right_on="gene",
#         how="inner"
#     )

#     if merged.shape[0] < 10:
#         return np.nan, np.nan, merged.shape[0]

#     rho, p = spearmanr(merged["deg_rank"], merged["pred_rank"])
#     return rho, p, merged.shape[0]

# # ============================
# # Collect correlations
# # ============================
# records = []

# # ---- Morabito ----
# morabito = pd.read_excel(
#     os.path.join(external_dir, "Morabito_DEG_results.xlsx"),
#     sheet_name="Supplementary Data 1c",
#     skiprows=2
# )

# mor_map = {
#     "ASC": "Ast", "MG": "Mic", "INH": "In",
#     "ODC": "Oli", "OPC": "Opc", "EX": "Ex"
# }

# for k, ct in mor_map.items():
#     rho, p, n = corr_abs_fc_spearman(
#         morabito[morabito["cluster"] == k], ct, "gene", "avg_logFC"
#     )
#     if not np.isnan(rho):
#         records.append(("Morabito", ct, rho, p, n))

# # ---- Sadick ----
# for ct, fname, up, down in [
#     ("Ast", "Sadick_Ast_DEG_results.xlsx",
#      "Astro_upregulated_DEGs_dis", "Astro_downregulated_DEGs_dis"),
#     ("Oli", "Sadick_Oli_DEG_results.xlsx",
#      "Oligo_upregulated_DEGs_dis", "Oligo_downregulated_DEGs_dis")
# ]:
#     dfs = []
#     for sh in [up, down]:
#         df = pd.read_excel(os.path.join(external_dir, fname), sheet_name=sh)
#         gcol = [c for c in df.columns if "gene" in c.lower()][0]
#         dfs.append(df[[gcol, "log2FC"]].rename(columns={gcol: "gene"}))
#     pooled = pd.concat(dfs).drop_duplicates("gene")

#     rho, p, n = corr_abs_fc_spearman(pooled, ct, "gene", "log2FC")
#     if not np.isnan(rho):
#         records.append(("Sadick", ct, rho, p, n))

# # ---- Su ----
# su = pd.read_excel(
#     os.path.join(external_dir, "Su_DEG_results.xlsx"),
#     sheet_name="TableS5A",
#     skiprows=1
# )

# su_map = {
#     "AST": "Ast", "MG": "Mic",
#     "ODC": "Oli", "OLIG": "Oli",
#     "OPC": "Opc", "Glut.N": "Ex",
#     "GABA.N": "In"
# }

# for k, ct in su_map.items():
#     rho, p, n = corr_abs_fc_spearman(
#         su[su["Cluster ID"] == k], ct, "Gene", "avg_logFC"
#     )
#     if not np.isnan(rho):
#         records.append(("Su", ct, rho, p, n))

# # ---- Grubman ----
# grub = pd.read_excel(
#     os.path.join(external_dir, "Grubman_DEG_results.xlsx"),
#     sheet_name="Supplementary Table 2",
#     header=5
# )

# grub_map = {
#     "mg": "Mic", "astro": "Ast",
#     "oligo": "Oli", "OPC": "Opc"
# }

# for k, ct in grub_map.items():
#     rho, p, n = corr_abs_fc_spearman(
#         grub[grub["group"] == k], ct, "geneID", "logFC"
#     )
#     if not np.isnan(rho):
#         records.append(("Grubman", ct, rho, p, n))

# # ============================
# # Final summaries + FDR
# # ============================
# df = pd.DataFrame(
#     records,
#     columns=["dataset", "cell_type", "rho", "p_value", "n_genes"]
# )

# df["p_value_fdr"] = multipletests(
#     df["p_value"], method="fdr_bh"
# )[1]

# celltype_summary = (
#     df.groupby("cell_type")
#       .agg(
#           mean_spearman_rho=("rho", "mean"),
#           n_datasets=("rho", "count")
#       )
#       .reset_index()
# )

# print(celltype_summary)
# print("\nEXTERNAL DEG VALIDATION.")

# # ============================
# # Write Excel ONLY
# # ============================
# description = pd.DataFrame({
#     "": [
#         "Analysis: External DEG vs Predictors (Mathys 48 (2019))",
#         "Statistic: Spearman rank correlation (DEG rank vs predictor rank)",
#         "Significance: asymptotic Spearman p-values, BH-FDR corrected",
#         "Predictors: stable predictors (nonzero importance in ≥2 of 5 splits)",
#         "DEGs: Morabito, Sadick, Su, Grubman",
#         "",
#     ]
# })

# with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl", mode="w") as writer:
#     description.to_excel(
#         writer,
#         sheet_name=SHEET_NAME,
#         index=False,
#         header=False,
#         startrow=0
#     )
#     celltype_summary.to_excel(
#         writer,
#         sheet_name=SHEET_NAME,
#         startrow=len(description) + 1,
#         index=False
#     )
#     df.to_excel(
#         writer,
#         sheet_name=SHEET_NAME,
#         startrow=len(description) + len(celltype_summary) + 4,
#         index=False
#     )

# print("\n✔ Sheet written:", SHEET_NAME)
# print("✔ Workbook:", EXCEL_OUT)

In [15]:
# # External gene-level vs Predictor validation — Spearman on |logFC|
# # All genes present in each external table are ranked (no DEG thresholding)

# import os
# import pandas as pd
# import numpy as np
# import joblib
# from scipy.stats import spearmanr
# from statsmodels.stats.multitest import multipletests

# # ============================
# # Paths
# # ============================
# external_dir = "/n/scratch/users/a/adm808/Revision/External_DEGs"
# model_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"

# SHEET_NAME = "External_vs_Predictor_abslogFC_ALLGENES"

# EXCEL_OUT = (
#     "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/"
#     f"Correlation_Analyses_{SHEET_NAME}.xlsx"
# )
# os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

# cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

# # ============================
# # Load predictors ONCE
# # ============================
# predictor_tables = {}
# MIN_SPLITS = 2

# for ct in cell_types:
#     split_imps = {}
#     genes = set()

#     for split in range(1, 6):
#         model = joblib.load(
#             os.path.join(model_dir, ct, f"split_{split}", "maximal_classifier.joblib")
#         )
#         split_imps[split] = dict(
#             zip(model.feature_names_in_, model.feature_importances_)
#         )
#         genes.update(model.feature_names_in_)

#     rows = []
#     for g in genes:
#         norm = []
#         nonzero_splits = 0

#         for s in split_imps:
#             imp = split_imps[s].get(g, 0.0)
#             denom = sum(v for v in split_imps[s].values() if v > 0)

#             if imp > 0:
#                 nonzero_splits += 1
#             norm.append(imp / denom if imp > 0 and denom > 0 else 0.0)

#         if nonzero_splits >= MIN_SPLITS:
#             rows.append({
#                 "gene": g,
#                 "mean_importance": np.mean(norm),
#                 "n_splits_nonzero": nonzero_splits
#             })

#     df = pd.DataFrame(rows)
#     df["pred_rank"] = df["mean_importance"].rank(ascending=False)
#     predictor_tables[ct] = df[["gene", "pred_rank", "n_splits_nonzero"]]

# # ============================
# # Helper — ALL GENES, ranked by |logFC|
# # ============================
# def corr_abs_logfc_allgenes(df, ct, gene_col, fc_col):
#     df = df.copy()
#     df[gene_col] = df[gene_col].astype(str).str.strip()
#     df[fc_col] = pd.to_numeric(df[fc_col], errors="coerce")
#     df = df.dropna(subset=[fc_col])

#     # Rank ALL rows by absolute logFC
#     # df["abs_logFC"] = df[fc_col].abs()
#     # df = df.sort_values("abs_logFC", ascending=False)
#     # df["effect_rank"] = np.arange(1, len(df) + 1)

#     df["abs_logFC"] = df[fc_col].abs()
#     df["effect_rank"] = df["abs_logFC"].rank(
#         ascending=False,
#         method="average"
#     )

#     merged = df.merge(
#         predictor_tables[ct],
#         left_on=gene_col,
#         right_on="gene",
#         how="inner"
#     )

#     if merged.shape[0] < 10:
#         return np.nan, np.nan, merged.shape[0]

#     rho, p = spearmanr(merged["effect_rank"], merged["pred_rank"])
#     return rho, p, merged.shape[0]

# # ============================
# # Collect correlations
# # ============================
# records = []

# # ---- Morabito ----
# morabito = pd.read_excel(
#     os.path.join(external_dir, "Morabito_DEG_results.xlsx"),
#     sheet_name="Supplementary Data 1c",
#     skiprows=2
# )

# mor_map = {
#     "ASC": "Ast", "MG": "Mic", "INH": "In",
#     "ODC": "Oli", "OPC": "Opc", "EX": "Ex"
# }

# for k, ct in mor_map.items():
#     rho, p, n = corr_abs_logfc_allgenes(
#         morabito[morabito["cluster"] == k], ct, "gene", "avg_logFC"
#     )
#     if not np.isnan(rho):
#         records.append(("Morabito", ct, rho, p, n))

# # ---- Sadick ----
# for ct, fname, up, down in [
#     ("Ast", "Sadick_Ast_DEG_results.xlsx",
#      "Astro_upregulated_DEGs_dis", "Astro_downregulated_DEGs_dis"),
#     ("Oli", "Sadick_Oli_DEG_results.xlsx",
#      "Oligo_upregulated_DEGs_dis", "Oligo_downregulated_DEGs_dis")
# ]:
#     dfs = []
#     for sh in [up, down]:
#         df = pd.read_excel(os.path.join(external_dir, fname), sheet_name=sh)
#         gcol = [c for c in df.columns if "gene" in c.lower()][0]
#         dfs.append(df[[gcol, "log2FC"]].rename(columns={gcol: "gene"}))
#     pooled = pd.concat(dfs).drop_duplicates("gene")

#     rho, p, n = corr_abs_logfc_allgenes(pooled, ct, "gene", "log2FC")
#     if not np.isnan(rho):
#         records.append(("Sadick", ct, rho, p, n))

# # ---- Su ----
# su = pd.read_excel(
#     os.path.join(external_dir, "Su_DEG_results.xlsx"),
#     sheet_name="TableS5A",
#     skiprows=1
# )

# su_map = {
#     "AST": "Ast", "MG": "Mic",
#     "ODC": "Oli", "OLIG": "Oli",
#     "OPC": "Opc", "Glut.N": "Ex",
#     "GABA.N": "In"
# }

# for k, ct in su_map.items():
#     rho, p, n = corr_abs_logfc_allgenes(
#         su[su["Cluster ID"] == k], ct, "Gene", "avg_logFC"
#     )
#     if not np.isnan(rho):
#         records.append(("Su", ct, rho, p, n))

# # ---- Grubman ----
# grub = pd.read_excel(
#     os.path.join(external_dir, "Grubman_DEG_results.xlsx"),
#     sheet_name="Supplementary Table 2",
#     header=5
# )

# grub_map = {
#     "mg": "Mic", "astro": "Ast",
#     "oligo": "Oli", "OPC": "Opc"
# }

# for k, ct in grub_map.items():
#     rho, p, n = corr_abs_logfc_allgenes(
#         grub[grub["group"] == k], ct, "geneID", "logFC"
#     )
#     if not np.isnan(rho):
#         records.append(("Grubman", ct, rho, p, n))

# # ============================
# # Final summaries + FDR
# # ============================
# df = pd.DataFrame(
#     records,
#     columns=["dataset", "cell_type", "rho", "p_value", "n_genes"]
# )

# df["p_value_fdr"] = multipletests(df["p_value"], method="fdr_bh")[1]

# celltype_summary = (
#     df.groupby("cell_type")
#       .agg(mean_spearman_rho=("rho", "mean"),
#            n_datasets=("rho", "count"))
#       .reset_index()
# )

# print(celltype_summary)
# print("\nEXTERNAL GENE-LEVEL VALIDATION.")

# # ============================
# # Write Excel ONLY
# # ============================
# description = pd.DataFrame({
#     "": [
#         "Analysis: External gene-level effect size vs Predictor importance",
#         "Statistic: Spearman rank correlation (|logFC| rank vs predictor rank)",
#         "Effect size: absolute log2 fold change (all genes reported per study)",
#         "Predictors: stable predictors (nonzero importance in ≥2 of 5 splits)",
#         "Datasets: Morabito, Sadick, Su, Grubman",
#         "",
#     ]
# })

# with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl", mode="w") as writer:
#     description.to_excel(
#         writer,
#         sheet_name=SHEET_NAME,
#         index=False,
#         header=False,
#         startrow=0
#     )
#     celltype_summary.to_excel(
#         writer,
#         sheet_name=SHEET_NAME,
#         startrow=len(description) + 1,
#         index=False
#     )
#     df.to_excel(
#         writer,
#         sheet_name=SHEET_NAME,
#         startrow=len(description) + len(celltype_summary) + 4,
#         index=False
#     )

# print("\n✔ Sheet written:", SHEET_NAME)
# print("✔ Workbook:", EXCEL_OUT)

In [16]:
# External gene-level vs Predictor validation
# Rank–rank Spearman: predictor rank vs |logFC| rank
# IDENTICAL logic to Mathys 427

import os
import pandas as pd
import numpy as np
import joblib
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# ============================
# Paths
# ============================
external_dir = "/n/scratch/users/a/adm808/Revision/External_DEGs"
model_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"

SHEET_NAME = "External_vs_Predictor_abslogFC"

EXCEL_OUT = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/"
    f"Correlation_Analyses_{SHEET_NAME}.xlsx"
)
os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

# ============================
# Load predictors (same as Mathys)
# ============================
predictor_tables = {}

for ct in cell_types:
    split_imps = {}
    all_genes = set()

    for split in range(1, 6):
        model = joblib.load(
            os.path.join(model_dir, ct, f"split_{split}", "maximal_classifier.joblib")
        )
        split_imps[split] = dict(
            zip(model.feature_names_in_, model.feature_importances_)
        )
        all_genes.update(model.feature_names_in_)

    rows = []
    for g in all_genes:
        norm_imps = []
        for s in split_imps:
            imp = split_imps[s].get(g, 0.0)
            denom = sum(v for v in split_imps[s].values() if v > 0)
            norm_imps.append(imp / denom if imp > 0 and denom > 0 else 0.0)

        rows.append({
            "gene": g,
            "mean_importance": np.mean(norm_imps)
        })

    imp_df = pd.DataFrame(rows)
    imp_df["pred_rank"] = imp_df["mean_importance"].rank(
        ascending=False, method="average"
    )
    predictor_tables[ct] = imp_df[["gene", "pred_rank"]]

# ============================
# Helper — rank by |logFC|
# ============================
def run_external_corr(df, ct, gene_col, fc_col):
    df = df.copy()
    df[gene_col] = df[gene_col].astype(str).str.strip()
    df[fc_col] = pd.to_numeric(df[fc_col], errors="coerce")
    df = df.dropna(subset=[fc_col])

    df["abs_logFC"] = df[fc_col].abs()
    df["deg_rank"] = df["abs_logFC"].rank(
        ascending=False, method="average"
    )

    merged = df.merge(
        predictor_tables[ct],
        left_on=gene_col,
        right_on="gene",
        how="inner"
    )

    if merged.shape[0] < 10:
        return np.nan, np.nan, merged.shape[0]

    rho, p = spearmanr(merged["deg_rank"], merged["pred_rank"])
    return rho, p, merged.shape[0]

# ============================
# Collect correlations
# ============================
records = []

# ---- Morabito ----
mor = pd.read_excel(
    os.path.join(external_dir, "Morabito_DEG_results.xlsx"),
    sheet_name="Supplementary Data 1c",
    skiprows=2
)

mor_map = {
    "ASC": "Ast", "MG": "Mic", "INH": "In",
    "ODC": "Oli", "OPC": "Opc", "EX": "Ex"
}

for k, ct in mor_map.items():
    rho, p, n = run_external_corr(
        mor[mor["cluster"] == k], ct, "gene", "avg_logFC"
    )
    records.append(("Morabito", ct, rho, p, n))

# ---- Sadick ----
for ct, fname, up, down in [
    ("Ast", "Sadick_Ast_DEG_results.xlsx",
     "Astro_upregulated_DEGs_dis", "Astro_downregulated_DEGs_dis"),
    ("Oli", "Sadick_Oli_DEG_results.xlsx",
     "Oligo_upregulated_DEGs_dis", "Oligo_downregulated_DEGs_dis")
]:
    dfs = []
    for sh in [up, down]:
        df = pd.read_excel(os.path.join(external_dir, fname), sheet_name=sh)
        gcol = [c for c in df.columns if "gene" in c.lower()][0]
        dfs.append(df[[gcol, "log2FC"]].rename(columns={gcol: "gene"}))
    pooled = pd.concat(dfs).drop_duplicates("gene")

    rho, p, n = run_external_corr(pooled, ct, "gene", "log2FC")
    records.append(("Sadick", ct, rho, p, n))

# ---- Su ----
su = pd.read_excel(
    os.path.join(external_dir, "Su_DEG_results.xlsx"),
    sheet_name="TableS5A",
    skiprows=1
)

su_map = {
    "AST": "Ast", "MG": "Mic",
    "ODC": "Oli", "OLIG": "Oli",
    "OPC": "Opc", "Glut.N": "Ex",
    "GABA.N": "In"
}

for k, ct in su_map.items():
    rho, p, n = run_external_corr(
        su[su["Cluster ID"] == k], ct, "Gene", "avg_logFC"
    )
    records.append(("Su", ct, rho, p, n))

# ---- Grubman ----
grub = pd.read_excel(
    os.path.join(external_dir, "Grubman_DEG_results.xlsx"),
    sheet_name="Supplementary Table 2",
    header=5
)

grub_map = {
    "mg": "Mic", "astro": "Ast",
    "oligo": "Oli", "OPC": "Opc"
}

for k, ct in grub_map.items():
    rho, p, n = run_external_corr(
        grub[grub["group"] == k], ct, "geneID", "logFC"
    )
    records.append(("Grubman", ct, rho, p, n))

# ============================
# Final tables
# ============================
df = pd.DataFrame(
    records,
    columns=["dataset", "cell_type", "spearman_rho", "p_value", "n_genes"]
)

df["p_value_fdr"] = multipletests(df["p_value"], method="fdr_bh")[1]

celltype_summary = (
    df.groupby("cell_type")
      .agg(mean_spearman_rho=("spearman_rho", "mean"),
           n_datasets=("spearman_rho", "count"))
      .reset_index()
)

print(celltype_summary)

# ============================
# Write Excel
# ============================
description = pd.DataFrame({
    "": [
        "Analysis: External predictor vs differential expression rank concordance",
        "Statistic: Spearman rank correlation between two ranked gene lists",
        "Predictor ranking: genes ranked by mean normalized feature importance, computed as follows:",
        "  (i) For each train/test split, raw feature importances are normalized by the sum of nonzero importances within that split;",
        "  (ii) Normalized importances are averaged across all splits;",
        "  (iii) Genes are ranked by this mean normalized importance (ties resolved by average rank).",
        "DEG ranking: genes ranked by absolute log fold change (|logFC| or |log2FC| as reported), with average-rank tie handling.",
        "Comparison set: intersection of genes present in both the external DEG table and the predictor set.",
        "Interpretation: assesses concordance between multivariate predictive prioritization and univariate differential expression ordering.",
        "Datasets: Morabito, Sadick, Su, Grubman.",
        "",
    ]
})

with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl", mode="w") as writer:
    description.to_excel(
        writer, sheet_name=SHEET_NAME,
        index=False, header=False, startrow=0
    )
    celltype_summary.to_excel(
        writer, sheet_name=SHEET_NAME,
        startrow=len(description) + 1,
        index=False
    )
    df.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        startrow=len(description) + len(celltype_summary) + 4,
        index=False
    )

print("\n✔ Sheet written:", SHEET_NAME)
print("✔ Workbook:", EXCEL_OUT)

  cell_type  mean_spearman_rho  n_datasets
0       Ast           0.087395           3
1        Ex           0.235064           2
2        In           0.054430           2
3       Mic           0.261018           3
4       Oli           0.181309           3
5       Opc           0.296532           3

✔ Sheet written: External_vs_Predictor_abslogFC
✔ Workbook: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/Correlation_Analyses_External_vs_Predictor_abslogFC.xlsx


In [17]:
# # Predictor vs Random Effect DEG for Mathys 427
# # Excel output only; logic unchanged

# import os
# import pandas as pd
# import numpy as np
# import joblib
# from scipy.stats import spearmanr
# from statsmodels.stats.multitest import multipletests

# # ============================
# # Paths
# # ============================
# deg_dir = "/n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed"
# model_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"

# SHEET_NAME = "Pred_vs_RE_DEG_Mathys427"

# EXCEL_OUT = (
#     "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/"
#     f"Correlation_Analyses_{SHEET_NAME}.xlsx"
# )
# os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

# cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

# results = []

# # ============================
# # Loop per cell type
# # ============================
# for ct in cell_types:
#     print(f"\n=== {ct} ===")

#     deg_path = os.path.join(deg_dir, f"poisson_DE_results_PFC_{ct}_COMBINED.csv")
#     deg = pd.read_csv(deg_path)

#     deg["gene"] = deg["gene"].str.strip()
#     deg = deg.sort_values("p_adj", ascending=True)
#     deg["deg_rank"] = np.arange(1, len(deg) + 1)

#     all_split_data = {}
#     all_features = set()

#     for split in range(1, 6):
#         model = joblib.load(
#             os.path.join(model_dir, ct, f"split_{split}", "maximal_classifier.joblib")
#         )

#         feats = model.feature_names_in_
#         imps = model.feature_importances_

#         all_features.update(feats)
#         all_split_data[split] = dict(zip(feats, imps))

#     rows = []
#     for gene in sorted(all_features):
#         norm_imps = []

#         for split in range(1, 6):
#             imp = all_split_data[split].get(gene, 0.0)
#             split_imps = np.array(list(all_split_data[split].values()))
#             denom = split_imps[split_imps > 0].sum()

#             norm_imps.append(imp / denom if imp > 0 and denom > 0 else 0.0)

#         rows.append({
#             "gene": gene.strip(),
#             "mean_importance": np.mean(norm_imps)
#         })

#     imp_df = pd.DataFrame(rows)
#     imp_df["pred_rank"] = imp_df["mean_importance"].rank(ascending=False)

#     merged = deg.merge(imp_df, on="gene", how="inner")

#     rho, p = spearmanr(merged["deg_rank"], merged["pred_rank"])

#     print(f"Spearman rho = {rho:.3f}, raw p = {p:.2e}")

#     results.append({
#         "cell_type": ct,
#         "spearman_rho": rho,
#         "p_value": p,
#         "n_genes": merged.shape[0]
#     })

# # ============================
# # Summary + FDR correction
# # ============================
# summary = pd.DataFrame(results)
# summary["p_value_fdr"] = multipletests(
#     summary["p_value"], method="fdr_bh"
# )[1]

# # ---------- PRINT FDR-ADJUSTED RESULTS ----------
# print("\n=== FDR-adjusted Predictor–DEG correlations ===")
# print(
#     summary[["cell_type", "spearman_rho", "p_value", "p_value_fdr", "n_genes"]]
# )

# print("\nAll correlations complete.")

# # ============================
# # Build wide-format table
# # ============================
# wide = pd.DataFrame(index=[
#     "Spearman rho",
#     "p-value",
#     "FDR-adjusted p-value",
#     "Number of genes"
# ])

# for ct in cell_types:
#     row = summary[summary["cell_type"] == ct].iloc[0]
#     wide[ct] = [
#         row["spearman_rho"],
#         row["p_value"],
#         row["p_value_fdr"],
#         row["n_genes"],
#     ]

# # ============================
# # Description block
# # ============================
# description = pd.DataFrame({
#     "": [
#         "Analysis: Predictor vs Random-effect DEG Correlation",
#         "Dataset: Mathys et al. (427 donors), PFC",
#         "DEG model: Random-effects Poisson differential expression",
#         "Predictors: mean normalized importance across 5 splits",
#         "Statistic: Spearman rank correlation (DEG rank vs predictor rank)",
#         "",
#     ]
# })

# # ============================
# # Write Excel sheet ONLY
# # ============================
# with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl", mode="w") as writer:
#     description.to_excel(
#         writer,
#         sheet_name=SHEET_NAME,
#         index=False,
#         header=False,
#         startrow=0
#     )

#     wide.to_excel(
#         writer,
#         sheet_name=SHEET_NAME,
#         startrow=len(description) + 1
#     )

# print("\n✔ Sheet written:", SHEET_NAME)
# print("✔ Workbook:", EXCEL_OUT)

In [18]:
# Predictor vs Random Effect DEG for Mathys 427
# Rank–rank Spearman: predictor rank vs |log2FC| rank
# Excel output only; prints preserved

import os
import pandas as pd
import numpy as np
import joblib
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# ============================
# Paths
# ============================
deg_dir = "/n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed"
model_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"

SHEET_NAME = "PredRank_vs_abslog2FC_REDEG_Mathys427"

EXCEL_OUT = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/"
    f"Correlation_Analyses_{SHEET_NAME}.xlsx"
)
os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

rows = []

# ============================
# Loop per cell type
# ============================
for ct in cell_types:
    print(f"\n=== {ct} ===")

    # ---------- Load DEG ----------
    deg_path = os.path.join(deg_dir, f"poisson_DE_results_PFC_{ct}_COMBINED.csv")
    deg = pd.read_csv(deg_path)

    deg["gene"] = deg["gene"].astype(str).str.strip()
    deg["log2FC"] = pd.to_numeric(deg["log2FC"], errors="coerce")
    deg = deg.dropna(subset=["log2FC"])

    # Rank by absolute log2FC
    # deg["abs_log2FC"] = deg["log2FC"].abs()
    # deg = deg.sort_values("abs_log2FC", ascending=False)
    # deg["deg_rank"] = np.arange(1, len(deg) + 1)

    deg["abs_log2FC"] = deg["log2FC"].abs()
    deg["deg_rank"] = deg["abs_log2FC"].rank(
        ascending=False,
        method="average"
    )

    tested_genes = set(deg["gene"])

    # ---------- Load predictors ----------
    split_imps = {}
    all_genes = set()

    for split in range(1, 6):
        model = joblib.load(
            os.path.join(model_dir, ct, f"split_{split}", "maximal_classifier.joblib")
        )
        split_imps[split] = dict(
            zip(model.feature_names_in_, model.feature_importances_)
        )
        all_genes.update(model.feature_names_in_)

    rows_imp = []
    for g in all_genes:
        g = str(g).strip()
        norm_imps = []

        for s in split_imps:
            imp = split_imps[s].get(g, 0.0)
            denom = sum(v for v in split_imps[s].values() if v > 0)
            norm_imps.append(imp / denom if imp > 0 and denom > 0 else 0.0)

        rows_imp.append({
            "gene": g,
            "mean_importance": np.mean(norm_imps)
        })

    imp_df = pd.DataFrame(rows_imp)
    imp_df = imp_df[imp_df["gene"].isin(tested_genes)].copy()

    # Rank predictors
    # imp_df = imp_df.sort_values("mean_importance", ascending=False)
    # imp_df["pred_rank"] = np.arange(1, len(imp_df) + 1)
    imp_df["pred_rank"] = imp_df["mean_importance"].rank(
    ascending=False,
    method="average")

    # ---------- Merge and correlate ----------
    merged = deg[["gene", "deg_rank"]].merge(
        imp_df[["gene", "pred_rank"]],
        on="gene",
        how="inner"
    )

    if merged.shape[0] < 10:
        rho, p = np.nan, np.nan
    else:
        rho, p = spearmanr(merged["deg_rank"], merged["pred_rank"])

    print(
        f"n_genes={merged.shape[0]} | "
        f"Spearman rho={rho:.3f} | raw p={p:.2e}"
    )

    rows.append({
        "cell_type": ct,
        "spearman_rho": rho,
        "p_value": p,
        "n_genes": merged.shape[0]
    })

# ============================
# Summary + FDR
# ============================
summary = pd.DataFrame(rows)
summary["p_value_fdr"] = multipletests(
    summary["p_value"], method="fdr_bh"
)[1]

print("\n=== FDR-adjusted Rank–Rank Predictor vs |log2FC| correlations (Mathys 427) ===")
print(
    summary[["cell_type", "spearman_rho", "p_value", "p_value_fdr", "n_genes"]]
)

# ============================
# Wide-format table
# ============================
wide = pd.DataFrame(index=[
    "Spearman rho",
    "p-value",
    "FDR-adjusted p-value",
    "Number of genes"
])

for ct in cell_types:
    row = summary[summary["cell_type"] == ct].iloc[0]
    wide[ct] = [
        row["spearman_rho"],
        row["p_value"],
        row["p_value_fdr"],
        row["n_genes"],
    ]

# ============================
# Description block
# ============================
description = pd.DataFrame({
    "": [
        "Analysis: Predictor vs random-effects DEG rank concordance",
        "Dataset: Mathys et al. (427 donors), PFC",
        "DEG model: Random-effects Poisson differential expression",
        "DEG ranking: absolute log2 fold change (all genes tested)",
        "Predictor ranking: mean normalized feature importance",
        "Statistic: Spearman rank correlation (rank–rank)",
        "Multiple testing: BH-FDR across cell types",
        "",
    ]
})

# ============================
# Write Excel ONLY
# ============================
with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl", mode="w") as writer:
    description.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        index=False,
        header=False,
        startrow=0
    )
    wide.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        startrow=len(description) + 1
    )

print("\n✔ Sheet written:", SHEET_NAME)
print("✔ Workbook:", EXCEL_OUT)


=== Ast ===
n_genes=2802 | Spearman rho=0.110 | raw p=5.58e-09

=== Mic ===
n_genes=1367 | Spearman rho=0.214 | raw p=1.13e-15

=== In ===
n_genes=5126 | Spearman rho=0.020 | raw p=1.52e-01

=== Oli ===
n_genes=1000 | Spearman rho=0.189 | raw p=1.64e-09

=== Opc ===
n_genes=3556 | Spearman rho=0.032 | raw p=5.35e-02

=== Ex ===
n_genes=7205 | Spearman rho=0.099 | raw p=3.17e-17

=== FDR-adjusted Rank–Rank Predictor vs |log2FC| correlations (Mathys 427) ===
  cell_type  spearman_rho       p_value   p_value_fdr  n_genes
0       Ast      0.109832  5.580516e-09  8.370775e-09     2802
1       Mic      0.214379  1.125703e-15  3.377109e-15     1367
2        In      0.019998  1.522631e-01  1.522631e-01     5126
3       Oli      0.189191  1.641679e-09  3.283358e-09     1000
4       Opc      0.032382  5.349983e-02  6.419980e-02     3556
5        Ex      0.099205  3.169498e-17  1.901699e-16     7205

✔ Sheet written: PredRank_vs_abslog2FC_REDEG_Mathys427
✔ Workbook: /n/groups/patel/adithya/Alz_O

/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


In [19]:
# Predictor vs DEG (Mathys 427) — overlap % + Fisher OR + FDR
# Excel output only; logic unchanged

import os
import numpy as np
import pandas as pd
import joblib
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

# ============================
# Paths
# ============================
deg_dir   = "/n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed"
model_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"

SHEET_NAME = "Pred_vs_DEG_Mathys427_Fisher"

EXCEL_OUT = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/"
    f"Correlation_Analyses_{SHEET_NAME}.xlsx"
)
os.makedirs(os.path.dirname(EXCEL_OUT), exist_ok=True)

cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

# Predictor definition
MIN_SPLITS = 2

# DEG definition
P_ADJ_CUTOFF = 0.05
ABS_LOG2FC_CUTOFF = 0.25

rows = []

# ============================
# Loop per cell type
# ============================
for ct in cell_types:
    print(f"\n=== {ct} ===")

    # ---------- Load 427 DEG ----------
    deg_path = os.path.join(deg_dir, f"poisson_DE_results_PFC_{ct}_COMBINED.csv")
    deg = pd.read_csv(deg_path)

    deg["gene"] = deg["gene"].astype(str).str.strip()
    deg["p_adj"] = pd.to_numeric(deg["p_adj"], errors="coerce")
    deg["log2FC"] = pd.to_numeric(deg["log2FC"], errors="coerce")

    # DEG-tested universe
    deg_tested = deg.dropna(subset=["p_adj", "log2FC"])
    tested_genes = set(deg_tested["gene"])

    # Significant DEGs
    deg_sig = deg_tested[
        (deg_tested["p_adj"] < P_ADJ_CUTOFF) &
        (deg_tested["log2FC"].abs() > ABS_LOG2FC_CUTOFF)
    ]
    deg_sig_genes = set(deg_sig["gene"])

    # ---------- Load predictors ----------
    gene_counts = {}
    for split in range(1, 6):
        model = joblib.load(
            os.path.join(model_dir, ct, f"split_{split}", "maximal_classifier.joblib")
        )
        for g, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                g = str(g).strip()
                gene_counts[g] = gene_counts.get(g, 0) + 1

    predictors = {g for g, c in gene_counts.items() if c >= MIN_SPLITS}

    # Restrict predictors to DEG-tested universe
    predictors_tested = predictors & tested_genes

    # Overlap
    overlap = predictors_tested & deg_sig_genes

    overlap_pct = (
        len(overlap) / len(predictors_tested)
        if len(predictors_tested) > 0 else np.nan
    )

    # ---------- Fisher exact test ----------
    a = len(overlap)
    b = len(predictors_tested) - a
    c = len(deg_sig_genes - predictors_tested)
    d = len(tested_genes) - (a + b + c)

    if d < 0:
        raise ValueError(f"Negative d for {ct}. Check universe definitions.")

    OR, p = fisher_exact([[a, b], [c, d]], alternative="two-sided")

    print(
        f"tested_genes={len(tested_genes)} | "
        f"predictors_tested={len(predictors_tested)} | "
        f"deg_sig={len(deg_sig_genes)}"
    )
    print(
        f"overlap={a} | overlap_pct={overlap_pct:.4f} | "
        f"OR={OR:.3f} | raw p={p:.2e}"
    )

    rows.append({
        "cell_type": ct,
        "n_tested_genes": len(tested_genes),
        "n_predictors_total": len(predictors),
        "n_predictors_tested": len(predictors_tested),
        "n_deg_sig": len(deg_sig_genes),
        "n_overlap": a,
        "overlap_pct_of_predictors_tested": overlap_pct,
        "odds_ratio": OR,
        "p_value": p
    })

# ============================
# Summary + FDR
# ============================
summary = pd.DataFrame(rows)
summary["p_value_fdr"] = multipletests(
    summary["p_value"], method="fdr_bh"
)[1]

print("\n=== FDR-adjusted Predictor–DEG (Mathys 427) overlap enrichment (Fisher) ===")
print(summary[[
    "cell_type",
    "n_predictors_tested",
    "n_deg_sig",
    "n_overlap",
    "overlap_pct_of_predictors_tested",
    "odds_ratio",
    "p_value",
    "p_value_fdr"
]])

# ============================
# Build wide-format table
# ============================
wide = pd.DataFrame(index=[
    "Tested genes (DEG universe)",
    "Predictors (total)",
    "Predictors (tested universe)",
    "Significant DEGs",
    "Overlap",
    "Overlap % of predictors (tested)",
    "Odds ratio (Fisher)",
    "p-value",
    "FDR-adjusted p-value"
])

for ct in cell_types:
    row = summary[summary["cell_type"] == ct].iloc[0]
    wide[ct] = [
        row["n_tested_genes"],
        row["n_predictors_total"],
        row["n_predictors_tested"],
        row["n_deg_sig"],
        row["n_overlap"],
        row["overlap_pct_of_predictors_tested"],
        row["odds_ratio"],
        row["p_value"],
        row["p_value_fdr"],
    ]

# ============================
# Description block
# ============================
description = pd.DataFrame({
    "": [
        "Analysis: Predictor vs DEG overlap enrichment",
        "Dataset: Mathys et al. (427 donors), PFC",
        "Predictors: stable predictors (nonzero importance in ≥2 of 5 splits)",
        "DEG definition: p_adj < 0.05 and |log2FC| > 0.25",
        "Universe: genes tested in DEG model",
        "Statistic: Fisher exact test (two-sided); FDR (BH) across cell types",
        "",
    ]
})

# ============================
# Write Excel ONLY
# ============================
with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl", mode="w") as writer:
    description.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        index=False,
        header=False,
        startrow=0
    )
    wide.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        startrow=len(description) + 1
    )

print("\n✔ Sheet written:", SHEET_NAME)
print("✔ Workbook:", EXCEL_OUT)


=== Ast ===
tested_genes=3894 | predictors_tested=158 | deg_sig=83
overlap=12 | overlap_pct=0.0759 | OR=4.243 | raw p=1.04e-04

=== Mic ===
tested_genes=2417 | predictors_tested=440 | deg_sig=15
overlap=9 | overlap_pct=0.0205 | OR=6.860 | raw p=3.60e-04

=== In ===
tested_genes=6145 | predictors_tested=102 | deg_sig=2
overlap=0 | overlap_pct=0.0000 | OR=0.000 | raw p=1.00e+00

=== Oli ===
tested_genes=2862 | predictors_tested=583 | deg_sig=29
overlap=12 | overlap_pct=0.0206 | OR=2.796 | raw p=9.08e-03

=== Opc ===
tested_genes=4697 | predictors_tested=788 | deg_sig=8
overlap=1 | overlap_pct=0.0013 | OR=0.708 | raw p=1.00e+00

=== Ex ===
tested_genes=9012 | predictors_tested=157 | deg_sig=40
overlap=4 | overlap_pct=0.0255 | OR=6.405 | raw p=4.97e-03

=== FDR-adjusted Predictor–DEG (Mathys 427) overlap enrichment (Fisher) ===
  cell_type  n_predictors_tested  n_deg_sig  n_overlap  \
0       Ast                  158         83         12   
1       Mic                  440         15    

In [20]:
# # Merging code

# import os
# import pandas as pd

# # ============================
# # INPUT WORKBOOKS
# # ============================
# input_files = [
#     "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/"
#     "Correlation_Analyses_DEG_Mathys2019_vs_Mathys2023_log2FC.xlsx",

#     "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/"
#     "Correlation_Analyses_Predictor_vs_RE_DEG_Mathys48.xlsx",

#     "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/"
#     "Correlation_Analyses_PredRank_Mathys48_vs_Skene.xlsx",
# ]

# # ============================
# # OUTPUT WORKBOOK
# # ============================
# OUT_EXCEL = (
#     "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/"
#     "Correlation_Analyses.xlsx"
# )

# os.makedirs(os.path.dirname(OUT_EXCEL), exist_ok=True)

# # ============================
# # Merge logic
# # ============================
# with pd.ExcelWriter(OUT_EXCEL, engine="openpyxl", mode="w") as writer:

#     for path in input_files:
#         if not os.path.exists(path):
#             raise FileNotFoundError(path)

#         xls = pd.ExcelFile(path)
#         if len(xls.sheet_names) != 1:
#             raise ValueError(f"{path} has {len(xls.sheet_names)} sheets; expected 1")

#         sheet = xls.sheet_names[0]
#         df = pd.read_excel(path, sheet_name=sheet, header=None)

#         df.to_excel(
#             writer,
#             sheet_name=sheet,
#             index=False,
#             header=False
#         )

# print("\n✔ Merged workbook written:")
# print("✔", OUT_EXCEL)

In [25]:
import os
import pandas as pd
from datetime import datetime

# ============================================================
# Directory containing ALL correlation analysis workbooks
# ============================================================
BASE_DIR = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/"
    "Revision/Tables/Correlations_New/"
)

OUT_EXCEL = os.path.join(
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/",
    "Correlation_Analyses.xlsx"
)

os.makedirs(os.path.dirname(OUT_EXCEL), exist_ok=True)

# ============================================================
# Explicit ordered file list (authoritative)
# ============================================================
ORDERED_FILES = [
    # ---- Core predictor vs DEG (FIRST) ----
    "Correlation_Analyses_Predictor_vs_RE_DEG_Mathys48.xlsx",
    "Correlation_Analyses_PredRank_vs_abslog2FC_REDEG_Mathys427.xlsx",
    "Correlation_Analyses_Pred_vs_DEG_Mathys427_Fisher.xlsx",
    "Correlation_Analyses_External_vs_Predictor_abslogFC.xlsx",

    # ---- 2024 analyses ----
    "Correlation_Analyses_PredRank_vs_abslog2FC_REDEG_Mathys2024.xlsx",
    "Correlation_Analyses_Pred_vs_DEG_Mathys2024_Fisher.xlsx",
    "Correlation_Analyses_DEG_Mathys2019_vs_Mathys2024_log2FC.xlsx",

    # ---- Predictor–predictor correlations ----
    "Correlation_Analyses_PredRank_Mathys48_vs_Lau.xlsx",
    "Correlation_Analyses_PredOverlap_Mathys48_vs_Lau.xlsx",
    "Correlation_Analyses_PredRank_Mathys48_vs_Skene.xlsx",
    "Correlation_Analyses_PredOverlap_Mathys48_vs_Skene.xlsx",

    # ---- FIXED EFFECT COMPARISONS (LAST) ----
    "Correlation_Analyses_DEG_RE_vs_Fixed_Mathys48_log2FC.xlsx",
    "Correlation_Analyses_DEG_RE_vs_FixedBatchApoe_Mathys48_log2FC.xlsx",
]

# ============================================================
# Helper: standardize sheet names ONLY in merged workbook
# ============================================================
def standardize_sheet_name(name):
    name = name.replace("Mathys2019", "Mathys48")
    name = name.replace("Mathys_2019", "Mathys48")
    name = name.replace("Mathys48", "Mathys48")
    name = name.replace("Mathys427", "Mathys427")
    name = name.replace("Mathys2023", "Mathys2023")
    name = name.replace("Mathys 2023", "Mathys2023")
    name = name.replace("Mathys 427", "Mathys427")
    return name[:31]  # Excel hard limit

# ============================================================
# Build master workbook
# ============================================================
with pd.ExcelWriter(OUT_EXCEL, engine="openpyxl", mode="w") as writer:

    # --------------------------------------------------------
    # 0) DESCRIPTION SHEET (FIRST)
    # --------------------------------------------------------
    description = pd.DataFrame({
        "": [
            "Correlation Analyses – Supplementary Tables",
            "",
            "This workbook aggregates all correlation and overlap analyses performed",
            "for our Alzheimer's disease prediction study.",
            "",
            "Analyses include:",
            "- Predictor vs random-effect DEG correlations",
            "- Predictor vs DEG overlap (Fisher exact tests)",
            "- Predictor importance vs DEG effect size correlations",
            "- External DEG validation",
            "- Predictor–predictor comparisons (Lau, Skene)",
            "- Random-effect vs fixed-effect DEG comparisons",
        ]
    })

    description.to_excel(
        writer,
        sheet_name="README",
        index=False,
        header=False
    )

    # --------------------------------------------------------
    # 1) Merge ordered analysis sheets
    # --------------------------------------------------------
    for fname in ORDERED_FILES:
        path = os.path.join(BASE_DIR, fname)

        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing expected file: {path}")

        xls = pd.ExcelFile(path)
        if len(xls.sheet_names) != 1:
            raise ValueError(f"{fname} has {len(xls.sheet_names)} sheets; expected exactly 1")

        orig_sheet = xls.sheet_names[0]
        new_sheet = standardize_sheet_name(orig_sheet)

        df = pd.read_excel(path, sheet_name=orig_sheet, header=None)

        df.to_excel(
            writer,
            sheet_name=new_sheet,
            index=False,
            header=False
        )

print("\n✔ Master Correlation_Analyses workbook written:")
print("✔", OUT_EXCEL)


✔ Master Correlation_Analyses workbook written:
✔ /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Correlations_New/Correlation_Analyses.xlsx
